In [ ]:
import pandas as pd
import requests
import os
import glob
import datetime as datetime
import pandas as pd
from dataIO import dataloader, webservices
from statisticscalculator import generalstatistics, climatestatistics
from plot_collection import stackedlineplots, streamflow_stat_plots, streamflow_stat_plots_matplotlib
import plotly.express as px

In [ ]:
# locations = pd.read_excel('locations_allMerged.xlsx')
# mf_flows = pd.read_excel('mf_flows_appendix_a.xlsx')
# headwater = pd.read_excel('headwater_sites.xlsx')

In [ ]:
headwater

In [ ]:
headwater = pd.read_excel('headwater_sites_revised.xlsx')
headwater.set_index('site_no', inplace=True)
site_list = list(headwater.index)
site_names = list(headwater['site_name'])
lats = list(headwater['lat'])
longs = list(headwater['long'])            

In [ ]:
site_names

In [ ]:
volume_stats_for_vip_sites = {}

# def vol_bw_dates():

for site in site_list:
    try:
        # basin = site[1][1]
        data = webservices.usgs_streamflow().get_data(start_date='1948-10-01', end_date='2021-10-01',sites=str(site)).reset_index()
        d = dataloader.DataLoader(data, 'Date', name_of_Q_column='Discharge')
        s = climatestatistics.Streamflow(d)
    except Exception as e:
        print(e)
    finally:
        pass
    
    try:
        mann_kendall_results = {}
        s.calc_max(calc_from_rolling_median=False, window_size=7)
        # print(f'Mann-Kendall Test Results looking for trend in day of year on peak runoff dates: {s.rolling_yr_Qmax_mk_test}')
        # print(f'Mann-Kendall Test Results looking for trend in discharge  on peak runoff dates: {s.rolling_yr_DOYmax_mk_test}')
    #     peak_runoff_plot = streamflow_stat_plots.Plot_Peak_Runoff(s)
    #     peak_runoff_plot = peak_runoff_plot.plot_peak_runoff(site[0])
        # print('-------------------------------------------------')  
    
        s.calc_annual_runoff_threshold_day(0.5, alpha=.05)
    #     volume_stats_for_vip_sites[site[1]] = s.threshold_vol_stats 
        # print(f'Mann-Kendall Test Results looking for trend in {s._percent*100}% runoff timing: {s.threshold_vol_mann_kendall_test}')
        # print(f'Mann-Kendall Test Results looking for trend in total annual discharge: {s.total_volume_mann_kendall_test}')  
    #     threshold_runoff_plot = streamflow_stat_plots.Plot_Runoff_Threshold(s)
    #     threshold_runoff_plot.plot_runoff_threshold(site[0])
    
    #     threshold_runoff_plot = streamflow_stat_plots_matplotlib.Plot_Runoff_Threshold(s)
    #     threshold_runoff_plot.plot_runoff_threshold(site[0])
        # print('-------------------------------------------------')
        
    #     for day in ['01-01', '03-01', '05-01', '08-01']:
        s.calc_runoff_bw_days(begin_month_day='01-01', end_month_day='12-31', alpha=.05)
        # print(f'Mann-Kendall Test Results looking for trend in discharge from {s.begin_month_day} to {s.end_month_day}: {s.volume_bw_days_mann_kendall_test}')
    #     runoff_bw_dates_plot = streamflow_stat_plots.Plot_Runoff_Volume_Between_2Days(s)
    #     runoff_bw_dates_plot.plot_runoff_volume_between_2days(site[0])
    
        # print('-------------------------------------------------')      
#         dates_ranges = {'01-01':'04-01', '04-01':'07-31', '08-01':'09-30', '10-01':'12-31'} 

        s.calc_runoff_bw_days(begin_month_day='10-01', end_month_day='12-31', alpha=.05)
        # print(f'Mann-Kendall Test Results looking for trend in discharge from {s.begin_month_day} to {s.end_month_day}: {s.volume_bw_days_mann_kendall_test}')
        runoff_bw_dates_plot = streamflow_stat_plots.Plot_Runoff_Volume_Between_2Days(s)
        # runoff_bw_dates_plot.plot_runoff_volume_between_2days(site[0])
        mann_kendall_results['Fall Vol']=s.volume_bw_days_mann_kendall_test
        # print('-------------------------------------------------')
    #     for day in ['01-01', '03-01', '05-01', '08-01']:
        s.calc_runoff_bw_days(begin_month_day='01-01', end_month_day='03-31', alpha=.05)
        print(f'Mann-Kendall Test Results looking for trend in discharge from {s.begin_month_day} to {s.end_month_day}: {s.volume_bw_days_mann_kendall_test}')
        runoff_bw_dates_plot = streamflow_stat_plots.Plot_Runoff_Volume_Between_2Days(s)
        # runoff_bw_dates_plot.plot_runoff_volume_between_2days(site[0])
        mann_kendall_results['Winter Vol']=s.volume_bw_days_mann_kendall_test
        print('-------------------------------------------------')
    #     for day in ['01-01', '03-01', '05-01', '08-01']:
        s.calc_runoff_bw_days(begin_month_day='04-01', end_month_day='06-30', alpha=.05)
        # print(f'Mann-Kendall Test Results looking for trend in discharge from {s.begin_month_day} to {s.end_month_day}: {s.volume_bw_days_mann_kendall_test}')
    #     runoff_bw_dates_plot2 = streamflow_stat_plots.Plot_Runoff_Volume_Between_2Days(s)
    #     runoff_bw_dates_plot2.plot_runoff_volume_between_2days(site[0])
        mann_kendall_results['Runoff Season Vol'] = s.volume_bw_days_mann_kendall_test
        print('-------------------------------------------------')            
    #     for day in ['01-01', '03-01', '05-01', '08-01']:
        s.calc_runoff_bw_days(begin_month_day='07-01', end_month_day='9-30', alpha=.05)
        # print(f'Mann-Kendall Test Results looking for trend in discharge from {s.begin_month_day} to {s.end_month_day}: {s.volume_bw_days_mann_kendall_test}')
    #     runoff_bw_dates_plot3 = streamflow_stat_plots.Plot_Runoff_Volume_Between_2Days(s)
        mann_kendall_results['Summer Vol'] = s.volume_bw_days_mann_kendall_test

    #     runoff_bw_dates_plot3.plot_runoff_volume_between_2days(site[0])
       
        
        mann_kendall_results['Q on date of peak Q date MK Test'] = s.rolling_yr_Qmax_mk_test
        mann_kendall_results['dayofyear on date of peak Q date MK Test'] = s.rolling_yr_DOYmax_mk_test
        mann_kendall_results['Threshold Vol MK Test'] = s.threshold_vol_mann_kendall_test
        mann_kendall_results['Threshold Vol DOY MK Test'] = s.threshold_vol_dates_mann_kendall_test
        mann_kendall_results['Total Vol MK Test'] = s.total_volume_mann_kendall_test
        mann_kendall_results['lat'] = headwater[headwater.index==site].iat[0,1]
        mann_kendall_results['long'] = headwater[headwater.index==site].iat[0,2]
        mann_kendall_results['name'] = headwater[headwater.index==site].iat[0,0]
        volume_stats_for_vip_sites[site] = mann_kendall_results
        
    except Exception as e:
        raise ValueError('Saving to volume_stats_for_vip_sites prob failed')
        print(e)
print('---------------------------------------------------------------------------------------------------')    


In [ ]:
sites = []
site_name = []
lat = []
long = []
# basins = []

PeakRunoffQ_trend = []
PeakRunoffQ_h = []
PeakRunoffQ_p = []
PeakRunoffQ_tau = []

PeakRunoffDOY_trend = []
PeakRunoffDOY_h = []
PeakRunoffDOY_p = []
PeakRunoffDOY_tau = []

ThresholdVol_trend = []
ThresholdVol_h = []
ThresholdVol_p = []
ThresholdVol_tau = []

ThresholdDOY_trend = []
ThresholdDOY_p = []
ThresholdDOY_tau = []

ThresholdQ_trend = []
ThresholdQ_h = []
ThresholdQ_p = []
ThresholdQ_tau = []

TotalVol_trend = []
TotalVol_h = []
TotalVol_p = []
TotalVol_tau = []

RunoffVol_trend = []
RunoffVol_h = []
RunoffVol_p = []
RunoffVol_tau = []

SummerVol_trend = []
SummerVol_h = []
SummerVol_p = []
SummerVol_tau = []

FallVol_trend = []
FallVol_h = []
FallVol_p = []
FallVol_tau = []

WinterVol_trend = []
WinterVol_h = []
WinterVol_p = []
WinterVol_tau = []

for site, value in volume_stats_for_vip_sites.items():
    
    sites.append(site)
    site_name.append(value.get('name'))
    lat.append(value.get('lat'))
    long.append(value.get('long'))              
                     
    # basins.append(site[1]['Basin'])
    
    PeakRunoffQ_trend.append(value.get('Q on date of peak Q date MK Test')[0])
    PeakRunoffQ_p.append(value.get('Q on date of peak Q date MK Test')[2])
    PeakRunoffQ_tau.append(value.get('Q on date of peak Q date MK Test')[4])
    
    PeakRunoffDOY_trend.append(value.get('Q on date of peak Q date MK Test')[0])
    PeakRunoffDOY_p.append(value.get('Q on date of peak Q date MK Test')[2])
    PeakRunoffDOY_tau.append(value.get('Q on date of peak Q date MK Test')[4])    
        
    ThresholdVol_trend.append(value.get('Threshold Vol MK Test')[0])
    ThresholdVol_p.append(value.get('Threshold Vol MK Test')[2])
    ThresholdVol_tau.append(value.get('Threshold Vol MK Test')[4])    
        
    ThresholdDOY_trend.append(value.get('Threshold Vol DOY MK Test')[0])
    ThresholdDOY_p.append(value.get('Threshold Vol DOY MK Test')[2])
    ThresholdDOY_tau.append(value.get('Threshold Vol DOY MK Test')[4])            
        
    TotalVol_trend.append(value.get('Total Vol MK Test')[0])
    TotalVol_p.append(value.get('Total Vol MK Test')[2])
    TotalVol_tau.append(value.get('Total Vol MK Test')[4])
    
    SummerVol_trend.append(value.get('Summer Vol')[0])
    SummerVol_p.append(value.get('Summer Vol')[2])
    SummerVol_tau.append(value.get('Summer Vol')[4])

    RunoffVol_trend.append(value.get('Runoff Season Vol')[0])
    RunoffVol_p.append(value.get('Runoff Season Vol')[2])
    RunoffVol_tau.append(value.get('Runoff Season Vol')[4])
    
    WinterVol_trend.append(value.get('Winter Vol')[0])
    WinterVol_p.append(value.get('Winter Vol')[2])
    WinterVol_tau.append(value.get('Winter Vol')[4])

    FallVol_trend.append(value.get('Fall Vol')[0])
    FallVol_p.append(value.get('Fall Vol')[2])
    FallVol_tau.append(value.get('Fall Vol')[4])
                                   
#     PeakRunoffMKTest['trend']=site[1]['Peak Runoff MK Test'][1]
#     PeakRunoffMKTest['h']=site[1]['Peak Runoff MK Test'][2]
#     PeakRunoffMKTest['p']=site[1]['Peak Runoff MK Test'][3]
#     PeakRunoffMKTest['Tau']=site[1]['Peak Runoff MK Test'][5]

In [ ]:
peak_runoffQ = pd.DataFrame({'sites':sites, 'site_name': site_name, 'lat': lat, 'long': long, 'trend': PeakRunoffQ_trend,  'p': PeakRunoffQ_p, 'tau': PeakRunoffQ_tau})
peak_runoffDOY = pd.DataFrame({'sites':sites, 'site_name': site_name, 'lat': lat, 'long': long, 'trend': PeakRunoffDOY_trend,  'p': PeakRunoffDOY_p, 'tau': PeakRunoffDOY_tau})
ThresholdVol = pd.DataFrame({'sites':sites, 'site_name': site_name, 'lat': lat, 'long': long, 'trend': ThresholdVol_trend, 'p': ThresholdVol_p, 'tau': ThresholdVol_tau})
ThresholdDOY = pd.DataFrame({'sites':sites, 'site_name': site_name, 'lat': lat, 'long': long, 'trend': ThresholdDOY_trend, 'p': ThresholdDOY_p, 'tau': ThresholdDOY_tau})
total_vol = pd.DataFrame({'sites':sites, 'site_name': site_name, 'lat': lat, 'long': long, 'trend': TotalVol_trend, 'p': TotalVol_p, 'tau': TotalVol_tau})
RunoffVol = pd.DataFrame({'sites':sites, 'site_name': site_name, 'lat': lat, 'long': long, 'trend': RunoffVol_trend, 'p': RunoffVol_p, 'tau': RunoffVol_tau})
SummerVol = pd.DataFrame({'sites':sites, 'site_name': site_name, 'lat': lat, 'long': long, 'trend': SummerVol_trend, 'p': SummerVol_p, 'tau': SummerVol_tau})
WinterVol = pd.DataFrame({'sites':sites, 'site_name': site_name, 'lat': lat, 'long': long, 'trend': WinterVol_trend, 'p': WinterVol_p, 'tau': WinterVol_tau})
FallVol = pd.DataFrame({'sites':sites, 'site_name': site_name, 'lat': lat, 'long': long, 'trend': FallVol_trend, 'p': FallVol_p, 'tau': FallVol_tau})

In [ ]:
peak_runoffQ_title = 'Trend in Peak Runoff Volume at Headwater Sites: Mann-Kendall Test Results' 
peak_runoffDOY_title = 'Trend in Timing During Peak Runoff at Headwater Sites: Mann-Kendall Test Results'
ThresholdDOY_title = 'Trends in Timing of 50% Yearly Volume at Headwater Sites: Mann-Kendall Test Results'
ThresholdVol_title = 'Trend in Volume at 50% Yearly Flow at Headwater Sites: Mann-Kendall Test Results'
total_vol_title = 'Yearly Total Volume Trends at Headwater Sites: Mann-Kendall Test Results'
SummerVol_title = 'Trend in Late Summer Volumes (08-01 through 09-30) at Headwater Sites: Mann-Kendall Test Results'
FallVol_title = 'Trend in Fall to early Winter Volumes (10-01 through 12-31) at Headwater Sites: Mann-Kendall Test Results'
WinterVol_title = 'Trend in Wintertime Volumes (01-01 through 4-01) at Headwater Sites: Mann-Kendall Test Results'
RunoffVol_title = 'Trend in Runoff Season Volumes (04-01 through 07-31) at Headwater Sites: Mann-Kendall Test Results'

In [ ]:

figs = []

fig = px.scatter(peak_runoffQ, 'p', 'tau', color='trend',
                 title=peak_runoffQ_title,
                 hover_data={
                     'Site Name': peak_runoffQ['site_name'],
                     'Site': peak_runoffQ['sites'], 'Trend': peak_runoffQ['trend'],
                 },
                 symbol = peak_runoffQ['trend'],
                 symbol_map={'no trend':'circle-open', 'decreasing': 'triangle-down', 'increasing':'triangle-up'},
                 color_discrete_map={'decreasing':'red', 'increasing':'green', 'no trend': 'blue'}
                )
figs.append(fig)
fig.show()

fig = px.scatter(peak_runoffDOY, 'p', 'tau', color='trend',
                 title=peak_runoffDOY_title,
                 hover_data={
                     'Site Name': peak_runoffDOY['site_name'],
                     'Site': peak_runoffDOY['sites'], 'Trend': peak_runoffDOY['trend'],},
                 symbol = peak_runoffDOY['trend'],
                 symbol_map={'no trend':'circle-open', 'decreasing': 'triangle-down', 'increasing':'triangle-up'},
                 color_discrete_map={'decreasing':'red', 'increasing':'green', 'no trend': 'blue'}
                )
figs.append(fig)


fig = px.scatter(ThresholdVol, 'p', 'tau', color='trend',
                 title=ThresholdVol_title,
                 hover_data={
                     'Site Name': ThresholdVol['site_name'],
                     'Site': ThresholdVol['sites'], 'Trend': ThresholdVol['trend'],},
                 symbol = ThresholdVol['trend'],
                 symbol_map={'no trend':'circle-open', 'decreasing': 'triangle-down', 'increasing':'triangle-up'},
                 color_discrete_map={'decreasing':'red', 'increasing':'green', 'no trend': 'blue'}
                )
figs.append(fig)
fig.show()

fig = px.scatter(ThresholdDOY, 'p', 'tau', color='trend',
                 title=ThresholdDOY_title,
                 hover_data={
                     'Site Name': ThresholdDOY['site_name'],
                     'Site': ThresholdDOY['sites'], 'Trend': ThresholdDOY['trend'],},
                 symbol = ThresholdDOY['trend'],
                 symbol_map={'no trend':'circle-open', 'decreasing': 'triangle-down', 'increasing':'triangle-up'},
                 color_discrete_map={'decreasing':'red', 'increasing':'green', 'no trend': 'blue'}
                )
figs.append(fig)
fig.show()

fig = px.scatter(total_vol, 'p', 'tau', color='trend',
                 title=total_vol_title,
                 hover_data={
                     'Site Name': total_vol['site_name'],
                     'Site': total_vol['sites'], 'Trend': total_vol['trend'],},
                 symbol = total_vol['trend'],
                 symbol_map={'no trend':'circle-open', 'decreasing': 'triangle-down', 'increasing':'triangle-up'},
                 color_discrete_map={'decreasing':'red', 'increasing':'green', 'no trend': 'blue'}
                )
figs.append(fig)
fig.show()

fig = px.scatter(SummerVol, 'p', 'tau', color='trend',
                 title=SummerVol_title,
                 hover_data={
                     'Site Name': SummerVol['site_name'], 
                     'Site': SummerVol['sites'], 'Trend': SummerVol['trend'],},
                 symbol = SummerVol['trend'],
                 symbol_map={'no trend':'circle-open', 'decreasing': 'triangle-down', 'increasing':'triangle-up'},
                 color_discrete_map={'decreasing':'red', 'increasing':'green', 'no trend': 'blue'}
                )
figs.append(fig)
fig.show()

fig = px.scatter(FallVol, 'p', 'tau', color='trend',
                 title=FallVol_title,
                 hover_data={
                     'Site Name': FallVol['site_name'], 
                     'Site': FallVol['sites'], 'Trend': FallVol['trend'],},
                 symbol = FallVol['trend'],
                 symbol_map={'no trend':'circle-open', 'decreasing': 'triangle-down', 'increasing':'triangle-up'},
                 color_discrete_map={'decreasing':'red', 'increasing':'green', 'no trend': 'blue'}
                )
figs.append(fig)
fig.show()

fig = px.scatter(WinterVol, 'p', 'tau', color='trend',
                 title=WinterVol_title,
                 hover_data={
                     'Site Name': WinterVol['site_name'], 
                     'Site': WinterVol['sites'], 'Trend': WinterVol['trend'],},
                 symbol = WinterVol['trend'],
                 symbol_map={'no trend':'circle-open', 'decreasing': 'triangle-down', 'increasing':'triangle-up'},
                 color_discrete_map={'decreasing':'red', 'increasing':'green', 'no trend': 'blue'}
                )
figs.append(fig)
fig.show()

fig = px.scatter(RunoffVol, 'p', 'tau', color='trend',
                 title=RunoffVol_title,
                 hover_data={
                     'Site Name': RunoffVol['site_name'], 
                     'Site': RunoffVol['sites'], 'Trend': RunoffVol['trend'],},
                 symbol = RunoffVol['trend'],
                 symbol_map={'no trend':'circle-open', 'decreasing': 'triangle-down', 'increasing':'triangle-up'},
                 color_discrete_map={'decreasing':'red', 'increasing':'green', 'no trend': 'blue'}
                )
figs.append(fig)
fig.show()

for fig in figs:
    with open('HeadwaterSites_MK_Results_1948to2021.html', 'a') as f:
        f.write(fig.to_html(full_html=False, include_plotlyjs='cdn', default_width=1200, default_height=400))


In [ ]:
!pip install geopandas

In [ ]:
import plotly.graph_objects as go
import pandas as pd
import geopandas as gpd
mapbox_token = "pk.eyJ1IjoiZ2Vv2tpbW90byIsImEi0iJja2NkcDBmcDEwMG5pMnJxcnQyMW01Z2djIn0.tei0yYjcWxex2zCtRjhzlA"

In [ ]:
figs = []
#      dates_ranges = {'01-01':'04-01', '04-01':'07-31', '08-01':'09-30', '10-01':'12-31'} 
for stat in [peak_runoffQ, peak_runoffDOY, ThresholdVol, ThresholdDOY, total_vol, RunoffVol, SummerVol, FallVol, WinterVol]:
    if stat is peak_runoffQ:
        title=peak_runoffQ_title
    elif stat is peak_runoffDOY:
        title=peak_runoffDOY_title
    elif stat is ThresholdVol:
        title=ThresholdVol_title
    elif stat is ThresholdDOY:
        title=ThresholdDOY_title
    elif stat is total_vol:
        title=total_vol_title
    elif stat is RunoffVol:
        title=RunoffVol_title
    elif stat is SummerVol:
        title=SummerVol_title
    elif stat is WinterVol:
        title=WinterVol_title
    elif stat is FallVol:
        title=FallVol_title
    

    # options = [['decreasing','circle'], ['no trend', 'circle-open'], ['increasing', 'circle']]
    trends = ['decreasing', 'no trend', 'increasing']
    markers = ['circle', 'circle-stroked', 'information']
    colors = ['red', 'grey', 'blue']

    
    # for option in options:
    #     print(option[0])
    #     print(option[1])
    fig = go.Figure()
    for trend, color in zip(trends, colors):
        # print(trend)
        # print(marker)
        
        fig.add_trace(
           go.Scattermapbox(
                mode='markers+text',
                lon = stat[stat["trend"]==trend]['long'],
                lat = stat[stat["trend"]==trend]['lat'],
                name = trend,
                text = stat[stat["trend"]==trend]['site_name'],
                textposition='top right',
                textfont=dict(
                    size=20,
                    color='black'
                ),
                customdata=stat[stat["trend"]==trend][['sites','p','tau','trend','site_name']],
                hovertemplate = 
                   '<b>%{text}</b><br>ID : %{customdata[0]}<br>Trend: %{customdata[3]}<br>P-value: %{customdata[1]:.2f}<br>tau: %{customdata[2]:.2f}<br><extra></extra>', # extra removes trace information
                             
                marker=go.scattermapbox.Marker(
                    size=10,
                    color=color,
                    # colorscale='Viridis',
                    # cmin=1,
                    # cmax=3,
                    # showscale=True
                    # symbol=marker, 
        )))

        fig.update_layout(
            
            mapbox_layers=[
                {
                    "below": 'traces',
                    "sourcetype": "raster",
                    "sourceattribution": "United States Geological Survey",
                    "source": [
                        "https://basemap.nationalmap.gov/arcgis/rest/services/USGSHydroCached/MapServer/tile/{z}/{y}/{x}"
                        # Options available - replace tag between services/ and /Mapserver with:
                        # USGSHydroCached
                        # USGSImageryOnly
                        # USGSImageryTopo
                        # USGSShadedReliefOnly
                        # USGSTNMBlank
                        # USGSTopo  
                    ],
                    # 'type':'raster',
                    # 'paint':{
                    #     'raster-opacity':0.5
                    #     },
                }
                # {
                #     "below": 'traces',
                #     "sourcetype": "raster",
                #     "sourceattribution": "United States Geological Survey",
                #     "source": [
                #         "https://basemap.nationalmap.gov/arcgis/rest/services/USGSShadedReliefOnly/MapServer/tile/{z}/{y}/{x}"
                #         # Options available - replace tag between services/ and /Mapserver with:
                #         # USGSHydroCached
                #         # USGSImageryOnly
                #         # USGSImageryTopo
                #         # USGSShadedReliefOnly
                #         # USGSTNMBlank
                #         # USGSTopo  
                #     ]       
                # }
            ],        
            mapbox = {
                # 'accesstoken': mapbox_token,
                'style': 'carto-positron', #'stamen-terrain',
                'zoom':5,
                'center': go.layout.mapbox.Center(lat=46.1, lon=-117.5)
            },
            title=f'<b>{title}</b>',
            showlegend=True,
            autosize=False,
            width=900,
            height=700
        )
    # file_name = f'{stat=}'.split('=')[0]
    # print(file_name)
    fig.show()
    figs.append(fig)
    
if os.path.exists('headwater_mk_results_maps.html'):
    os.remove('headwater_mk_results_maps.html')
for figure in figs:
    with open('headwater_mk_results_maps_1948to2021.html', 'a') as f:
        f.write(figure.to_html(full_html=False, include_plotlyjs='cdn', default_width=1200, default_height=400))

In [ ]:
site_list

# Decadal Trend Analysis

In [ ]:
headwater = pd.read_excel('headwater_sites.xlsx')
headwater.set_index('site_no', inplace=True)
site_list = list(headwater.index)
lats = list(headwater['lat'])
longs = list(headwater['long']) 

all_site_data = {}
for site in site_list:
    try:
        # basin = site[1][1]
        data = webservices.usgs_streamflow().get_data(start_date='1928-10-01', end_date='2023-10-01',sites=str(site)).reset_index()
        all_site_data[site] = data
    except Exception as e:
        print(e)
    finally:
        pass
    

In [ ]:
all_site_data.get(12301250).head(2)

In [ ]:
for start_date, end_date in dates.items():
    start_date = pd.to_datetime(start_date).date()
    end_date = pd.to_datetime(end_date).date()
    filtered_all_site_data = {}
    try:
        for k,v in all_site_data.items():
        # modified_flows_m_filtered_dates[k] = modified_flows_m.get(k)[modified_flows_m.get(k)['date']>=start_date]
            filtered_all_site_data[k] = all_site_data.get(k)[all_site_data.get(k)['Date'].between(start_date, end_date)]
            print(filtered_all_site_data)
    except Exception as e:
        print(e)
        pass

In [ ]:
dates = {
    '1928-10-01':'1938-10-01', '1938-10-01':'1948-10-01', '1948-10-01':'1958-10-01', '1958-10-01':'1968-10-01', '1968-10-01':'1978-10-01',
    # '1978-10-01':'1988-10-01', '1988-10-01':'1998-10-01', '1998-10-01':'2008-10-01', '2008-10-01':'2018-10-01', '2018-10-01':'2023-10-01'
    # '1996-10-01':'2010-10-01'
}




for start_date, end_date in dates.items():
    try:
        start_date = pd.to_datetime(start_date).date()
        end_date = pd.to_datetime(end_date).date()
        filtered_all_site_data = {}
        for k,v in all_site_data.items():
        # modified_flows_m_filtered_dates[k] = modified_flows_m.get(k)[modified_flows_m.get(k)['date']>=start_date]
            filtered_all_site_data[k] = all_site_data.get(k)[all_site_data.get(k)['Date'].between(start_date, end_date)]
    
    except Exception as e:
        print(e)
        pass

        
    mann_kendall_results = {}
    for site, df in filtered_all_site_data.items():
        try:
            print(site)
            d = dataloader.DataLoader(site[1], 'Date', name_of_Q_column='Discharge')
            s = climatestatistics.Streamflow(d)
    
                
            s.calc_max(calc_from_rolling_median=False, window_size=7)
            s.calc_annual_runoff_threshold_day(0.5, alpha=.05)
            s.calc_runoff_bw_days(begin_month_day='01-01', end_month_day='12-31', alpha=.05)
            s.calc_runoff_bw_days(begin_month_day='10-01', end_month_day='12-31', alpha=.05)
            runoff_bw_dates_plot = streamflow_stat_plots.Plot_Runoff_Volume_Between_2Days(s)
            # runoff_bw_dates_plot.plot_runoff_volume_between_2days(site[0])
            mann_kendall_results['Fall Vol']=s.volume_bw_days_mann_kendall_test
            # print('-------------------------------------------------')
        #     for day in ['01-01', '03-01', '05-01', '08-01']:
            s.calc_runoff_bw_days(begin_month_day='01-01', end_month_day='03-31', alpha=.05)
            # print(f'Mann-Kendall Test Results looking for trend in discharge from {s.begin_month_day} to {s.end_month_day}: {s.volume_bw_days_mann_kendall_test}')
            runoff_bw_dates_plot = streamflow_stat_plots.Plot_Runoff_Volume_Between_2Days(s)
            # runoff_bw_dates_plot.plot_runoff_volume_between_2days(site[0])
            mann_kendall_results['Winter Vol']=s.volume_bw_days_mann_kendall_test
            # print('-------------------------------------------------')
        #     for day in ['01-01', '03-01', '05-01', '08-01']:
            s.calc_runoff_bw_days(begin_month_day='04-01', end_month_day='06-30', alpha=.05)
            # print(f'Mann-Kendall Test Results looking for trend in discharge from {s.begin_month_day} to {s.end_month_day}: {s.volume_bw_days_mann_kendall_test}')
        #     runoff_bw_dates_plot2 = streamflow_stat_plots.Plot_Runoff_Volume_Between_2Days(s)
        #     runoff_bw_dates_plot2.plot_runoff_volume_between_2days(site[0])
            mann_kendall_results['Runoff Season Vol'] = s.volume_bw_days_mann_kendall_test
            # print('-------------------------------------------------')            
        #     for day in ['01-01', '03-01', '05-01', '08-01']:
            s.calc_runoff_bw_days(begin_month_day='07-01', end_month_day='9-30', alpha=.05)
            mann_kendall_results['Summer Vol'] = s.volume_bw_days_mann_kendall_test     
            mann_kendall_results['Q on date of peak Q date MK Test'] = s.rolling_yr_Qmax_mk_test
            mann_kendall_results['dayofyear on date of peak Q date MK Test'] = s.rolling_yr_DOYmax_mk_test
            mann_kendall_results['Threshold Vol MK Test'] = s.threshold_vol_mann_kendall_test
            mann_kendall_results['Threshold Vol DOY MK Test'] = s.threshold_vol_dates_mann_kendall_test
            mann_kendall_results['Total Vol MK Test'] = s.total_volume_mann_kendall_test
            # mann_kendall_results['Basin'] = basin
            mann_kendall_results['lat'] = headwater[headwater.index==site].iat[0,1]
            mann_kendall_results['long'] = headwater[headwater.index==site].iat[0,2]
            mann_kendall_results['name'] = headwater[headwater.index==site].iat[0,0]
            volume_stats_for_vip_sites[site] = mann_kendall_results
            
        except Exception as e:
            # raise ValueError('Saving to volume_stats_for_vip_sites prob failed')
            print(e)
        # finally:
        #     pass
    
    print(mann_kendall_results)
    
    sites = []
    site_name = []
    lat = []
    long = []
    # basins = []
    
    PeakRunoffQ_trend = []
    PeakRunoffQ_h = []
    PeakRunoffQ_p = []
    PeakRunoffQ_tau = []
    
    PeakRunoffDOY_trend = []
    PeakRunoffDOY_h = []
    PeakRunoffDOY_p = []
    PeakRunoffDOY_tau = []
    
    ThresholdVol_trend = []
    ThresholdVol_h = []
    ThresholdVol_p = []
    ThresholdVol_tau = []
    
    ThresholdDOY_trend = []
    ThresholdDOY_p = []
    ThresholdDOY_tau = []
    
    ThresholdQ_trend = []
    ThresholdQ_h = []
    ThresholdQ_p = []
    ThresholdQ_tau = []
    
    TotalVol_trend = []
    TotalVol_h = []
    TotalVol_p = []
    TotalVol_tau = []
    
    RunoffVol_trend = []
    RunoffVol_h = []
    RunoffVol_p = []
    RunoffVol_tau = []
    
    SummerVol_trend = []
    SummerVol_h = []
    SummerVol_p = []
    SummerVol_tau = []
    
    FallVol_trend = []
    FallVol_h = []
    FallVol_p = []
    FallVol_tau = []
    
    WinterVol_trend = []
    WinterVol_h = []
    WinterVol_p = []
    WinterVol_tau = []
    
    for site, value in volume_stats_for_vip_sites.items():
        
        sites.append(site)
        site_name.append(value.get('name'))
        lat.append(value.get('lat'))
        long.append(value.get('long'))              
                         
        # basins.append(site[1]['Basin'])
        
        PeakRunoffQ_trend.append(value.get('Q on date of peak Q date MK Test')[0])
        PeakRunoffQ_p.append(value.get('Q on date of peak Q date MK Test')[2])
        PeakRunoffQ_tau.append(value.get('Q on date of peak Q date MK Test')[4])
        
        PeakRunoffDOY_trend.append(value.get('Q on date of peak Q date MK Test')[0])
        PeakRunoffDOY_p.append(value.get('Q on date of peak Q date MK Test')[2])
        PeakRunoffDOY_tau.append(value.get('Q on date of peak Q date MK Test')[4])    
            
        ThresholdVol_trend.append(value.get('Threshold Vol MK Test')[0])
        ThresholdVol_p.append(value.get('Threshold Vol MK Test')[2])
        ThresholdVol_tau.append(value.get('Threshold Vol MK Test')[4])    
            
        ThresholdDOY_trend.append(value.get('Threshold Vol DOY MK Test')[0])
        ThresholdDOY_p.append(value.get('Threshold Vol DOY MK Test')[2])
        ThresholdDOY_tau.append(value.get('Threshold Vol DOY MK Test')[4])            
            
        TotalVol_trend.append(value.get('Total Vol MK Test')[0])
        TotalVol_p.append(value.get('Total Vol MK Test')[2])
        TotalVol_tau.append(value.get('Total Vol MK Test')[4])
        
        SummerVol_trend.append(value.get('Summer Vol')[0])
        SummerVol_p.append(value.get('Summer Vol')[2])
        SummerVol_tau.append(value.get('Summer Vol')[4])
    
        RunoffVol_trend.append(value.get('Runoff Season Vol')[0])
        RunoffVol_p.append(value.get('Runoff Season Vol')[2])
        RunoffVol_tau.append(value.get('Runoff Season Vol')[4])
        
        WinterVol_trend.append(value.get('Winter Vol')[0])
        WinterVol_p.append(value.get('Winter Vol')[2])
        WinterVol_tau.append(value.get('Winter Vol')[4])
    
        FallVol_trend.append(value.get('Fall Vol')[0])
        FallVol_p.append(value.get('Fall Vol')[2])
        FallVol_tau.append(value.get('Fall Vol')[4])
    
    
    
    peak_runoffQ = pd.DataFrame({'sites':sites, 'site_name': site_name, 'lat': lat, 'long': long, 'trend': PeakRunoffQ_trend,  'p': PeakRunoffQ_p, 'tau': PeakRunoffQ_tau})
    peak_runoffDOY = pd.DataFrame({'sites':sites, 'site_name': site_name, 'lat': lat, 'long': long, 'trend': PeakRunoffDOY_trend,  'p': PeakRunoffDOY_p, 'tau': PeakRunoffDOY_tau})
    ThresholdVol = pd.DataFrame({'sites':sites, 'site_name': site_name, 'lat': lat, 'long': long, 'trend': ThresholdVol_trend, 'p': ThresholdVol_p, 'tau': ThresholdVol_tau})
    ThresholdDOY = pd.DataFrame({'sites':sites, 'site_name': site_name, 'lat': lat, 'long': long, 'trend': ThresholdDOY_trend, 'p': ThresholdDOY_p, 'tau': ThresholdDOY_tau})
    total_vol = pd.DataFrame({'sites':sites, 'site_name': site_name, 'lat': lat, 'long': long, 'trend': TotalVol_trend, 'p': TotalVol_p, 'tau': TotalVol_tau})
    RunoffVol = pd.DataFrame({'sites':sites, 'site_name': site_name, 'lat': lat, 'long': long, 'trend': RunoffVol_trend, 'p': RunoffVol_p, 'tau': RunoffVol_tau})
    SummerVol = pd.DataFrame({'sites':sites, 'site_name': site_name, 'lat': lat, 'long': long, 'trend': SummerVol_trend, 'p': SummerVol_p, 'tau': SummerVol_tau})
    WinterVol = pd.DataFrame({'sites':sites, 'site_name': site_name, 'lat': lat, 'long': long, 'trend': WinterVol_trend, 'p': WinterVol_p, 'tau': WinterVol_tau})
    FallVol = pd.DataFrame({'sites':sites, 'site_name': site_name, 'lat': lat, 'long': long, 'trend': FallVol_trend, 'p': FallVol_p, 'tau': FallVol_tau})
    
    peak_runoffQ_title = f'Trend in Peak Runoff Volume at Headwater Sites: MK Results {start_date[:4]} to {end_date[:4]}' 
    peak_runoffDOY_title = f'Trend in Timing During Peak at Headwater Sites: MK Results {start_date[:4]} to {end_date[:4]}'
    ThresholdDOY_title = f'Trends in Timing of 50% Yearly Volume at Headwater Sites: MK Results {start_date[:4]} to {end_date[:4]}'
    ThresholdVol_title = f'Trend in Volume at 50% Yearly Flow at Headwater Sites: MK Results {start_date[:4]} to {end_date[:4]}'
    total_vol_title = f'Yearly Total Volume Trends at Sites at Headwater Sites: MK Results {start_date[:4]} to {end_date[:4]}'
    SummerVol_title = f'Trend in Late Summer Volumes (08-01 through 09-30) at Headwater Sites: MK Results {start_date[:4]} to {end_date[:4]}'
    FallVol_title = f'Trend in Fall to early Winter Volumes (10-01 through 12-31) at Headwater Sites: MK Results {start_date[:4]} to {end_date[:4]}'
    WinterVol_title = f'Trend in Wintertime Volumes (01-01 through 4-01)  at Headwater Sites: MK Results {start_date[:4]} to {end_date[:4]}'
    RunoffVol_title = f'Trend in Runoff Season Volumes (04-01 through 07-31) at Headwater Sites: MK Results {start_date[:4]} to {end_date[:4]}'
    
    
    figs = []
    fig = px.scatter(peak_runoffQ, 'p', 'tau', color='trend',
                     title=peak_runoffQ_title,
                     hover_data={
                         'Site Name': peak_runoffQ['site_name'], 
                         'Site': peak_runoffQ['sites'], 'Trend': peak_runoffQ['trend'],},
                     symbol = peak_runoffQ['trend'],
                     symbol_map={'no trend':'circle-open', 'decreasing': 'triangle-down', 'increasing':'triangle-up'},
                     color_discrete_map={'decreasing':'red', 'increasing':'green', 'no trend': 'blue'}
    
                    )
    figs.append(fig)
    fig.show()
    
    fig = px.scatter(peak_runoffDOY, 'p', 'tau', color='trend',
                     title=peak_runoffDOY_title,
                     hover_data={
                         'Site Name': peak_runoffDOY['site_name'], 
                         'Site': peak_runoffDOY['sites'], 'Trend': peak_runoffDOY['trend'],},
                     symbol = peak_runoffDOY['trend'],
                     symbol_map={'no trend':'circle-open', 'decreasing': 'triangle-down', 'increasing':'triangle-up'},
                     color_discrete_map={'decreasing':'red', 'increasing':'green', 'no trend': 'blue'}
    
                    )
    figs.append(fig)
    fig.show()
    
    
    fig = px.scatter(ThresholdVol, 'p', 'tau', color='trend',
                     title=ThresholdVol_title,
                     hover_data={
                         'Site Name': ThresholdVol['site_name'], 
                         'Site': ThresholdVol['sites'], 'Trend': ThresholdVol['trend'],},
                     symbol = ThresholdVol['trend'],
                     symbol_map={'no trend':'circle-open', 'decreasing': 'triangle-down', 'increasing':'triangle-up'},
                     color_discrete_map={'decreasing':'red', 'increasing':'green', 'no trend': 'blue'}
                    )
    figs.append(fig)
    fig.show()
    fig = px.scatter(ThresholdDOY, 'p', 'tau', color='trend',
                     title=ThresholdDOY_title,
                     hover_data={
                         'Site Name': ThresholdDOY['site_name'], 
                         'Site': ThresholdDOY['sites'], 'Trend': ThresholdDOY['trend'],},
                     symbol = ThresholdDOY['trend'],
                     symbol_map={'no trend':'circle-open', 'decreasing': 'triangle-down', 'increasing':'triangle-up'},
                     color_discrete_map={'decreasing':'red', 'increasing':'green', 'no trend': 'blue'}
                    )
    figs.append(fig)
    fig.show()
    # with open('HeadwaterSites_MK_Results2.html', 'a') as f:
    #     f.write(fig.to_html(full_html=False, include_plotlyjs='cdn', default_width=1200, default_height=400))
    
    fig = px.scatter(total_vol, 'p', 'tau', color='trend',
                     title=total_vol_title,
                     hover_data={
                         'Site Name': total_vol['site_name'], 
                         'Site': total_vol['sites'], 'Trend': total_vol['trend'],},
                     symbol = total_vol['trend'],
                     symbol_map={'no trend':'circle-open', 'decreasing': 'triangle-down', 'increasing':'triangle-up'},
                     color_discrete_map={'decreasing':'red', 'increasing':'green', 'no trend': 'blue'}
                    )
    figs.append(fig)
    fig.show()
    
    fig = px.scatter(SummerVol, 'p', 'tau', color='trend',
                     title=SummerVol_title,
                     hover_data={
                         'Site Name': SummerVol['site_name'], 
                         'Site': SummerVol['sites'], 'Trend': SummerVol['trend'],},
                     symbol = SummerVol['trend'],
                     symbol_map={'no trend':'circle-open', 'decreasing': 'triangle-down', 'increasing':'triangle-up'},
                     color_discrete_map={'decreasing':'red', 'increasing':'green', 'no trend': 'blue'}
                    )
    figs.append(fig)
    fig.show()
    
    fig = px.scatter(FallVol, 'p', 'tau', color='trend',
                     title=FallVol_title,
                     hover_data={
                         'Site Name': FallVol['site_name'], 
                         'Site': FallVol['sites'], 'Trend': FallVol['trend'],},
                     symbol = FallVol['trend'],
                     symbol_map={'no trend':'circle-open', 'decreasing': 'triangle-down', 'increasing':'triangle-up'},
                     color_discrete_map={'decreasing':'red', 'increasing':'green', 'no trend': 'blue'}
                    )
    figs.append(fig)
    fig.show()
    
    fig = px.scatter(WinterVol, 'p', 'tau', color='trend',
                     title=WinterVol_title,
                     hover_data={
                         'Site Name': WinterVol['site_name'], 
                         'Site': WinterVol['sites'], 'Trend': WinterVol['trend'],},
                     symbol = WinterVol['trend'],
                     symbol_map={'no trend':'circle-open', 'decreasing': 'triangle-down', 'increasing':'triangle-up'},
                     color_discrete_map={'decreasing':'red', 'increasing':'green', 'no trend': 'blue'}
                    )
    figs.append(fig)
    fig.show()
    
    fig = px.scatter(RunoffVol, 'p', 'tau', color='trend',
                     title=RunoffVol_title,
                     hover_data={
                         'Site Name': RunoffVol['site_name'], 
                         'Site': RunoffVol['sites'], 'Trend': RunoffVol['trend'],},
                     symbol = RunoffVol['trend'],
                     symbol_map={'no trend':'circle-open', 'decreasing': 'triangle-down', 'increasing':'triangle-up'},
                     color_discrete_map={'decreasing':'red', 'increasing':'green', 'no trend': 'blue'}
                    )
    figs.append(fig)
    fig.show()
    
    file_name = f'Headwater_MK_Results_{start_date[:4]}to{end_date[:4]}.html'
    if os.path.exists(file_name):
         os.remove(file_name)
    for fig in figs:       
        with open(file_name, 'a') as f:
            f.write(fig.to_html(full_html=False, include_plotlyjs='cdn', default_width=1200, default_height=400))


## Making into classes

In [ ]:
import pandas as pd
import requests
import os
import glob
import datetime as datetime
import pandas as pd
from dataIO import dataloader, webservices
from statisticscalculator import generalstatistics, climatestatistics
from plot_collection import stackedlineplots, streamflow_stat_plots, streamflow_stat_plots_matplotlib
import plotly.express as px

In [ ]:
class annual_stats():
    def __init__(self, StreamflowClimateStatistics):
        self.s = StreamflowClimateStatistics
        self.vol_mk_stats = {}
        self.runoff_timing_mk_stats = {}
        self.vol_figs = {}
        self.runoff_timing_figs = {}
        self.site_meta_data = {}
    
    def calc_stats(self,
        start_date='1928-10-01', 
        end_date='2023-10-01', 
        month_intervals={'01-01':'12-31', '01-01':'03-31', '04-01':'07-31', '08-01':'9-30', '10-01':'12-31'}
    ):
        # self.s.calc_max(calc_from_rolling_median=False, window_size=7)
        try:
            self.s.calc_annual_runoff_threshold_day(0.5, alpha=.05)
        except: # Exception as e:
            # print(e)
            pass
        for k,v in month_intervals.items():
            try:
                self.s.calc_runoff_bw_days(begin_month_day=k, end_month_day=v, alpha=0.05)
                # self.vol_figs[f'{k}_to_{v}'] = streamflow_stat_plots.Plot_Runoff_Volume_Between_2Days(s)
                self.vol_mk_stats[f'{k}_to_{v}'] = s.volume_bw_days_mann_kendall_test
            except Exception as e:
                print('Issue occurred calculating volume stats or creating figures.')
                print(e)
                pass
        try:
            self.vol_mk_stats['Total Vol MK Test'] = s.total_volume_mann_kendall_test
            
            # self.runoff_timing_mk_stats['Q on date of peak Q date MK Test'] = s.rolling_yr_Qmax_mk_test
            # self.runoff_timing_mk_stats['dayofyear on date of peak Q date MK Test'] = s.rolling_yr_DOYmax_mk_test
            self.runoff_timing_mk_stats['Threshold Vol MK Test'] = s.threshold_vol_mann_kendall_test
            self.runoff_timing_mk_stats['Threshold Vol DOY MK Test'] = s.threshold_vol_dates_mann_kendall_test
            
            self.site_meta_data['lat'] = headwater[headwater.index==site].iat[0,1]
            self.site_meta_data['long'] = headwater[headwater.index==site].iat[0,2]
            self.site_meta_data['name'] = headwater[headwater.index==site].iat[0,0]
            self.site_meta_data['site'] = site
        except Exception as e:
            print('Error occurred calculating mann-kendall on total volume, runoff timing and volume, or adding site meta data')
            print(e)

class aggregate_stats():
    
    def __init__(self, annual_stats_objs):
        self.annual_stats_objs = annual_stats_objs
        self.aggregate_peakrunoffQ = {}
        self.aggregate_peakrunoffDOY = {}
        self.aggregate_thresholdVol = {}
        self.aggregate_thresholdDOY = {}
        self.aggregate_vols = {}

        self.aggregate_peakrunoffQ_dfs = {}
        self.aggregate_peakrunoffDOY_dfs = {}
        self.aggregate_vols_dfs = {}
        self.figs = {}
        self.aggregate_meta_data = {}

    def _aggregate_stats(self):

        # for key in self.annual_stats_objs.get(.vol_mk_stats.keys():
        #     print(key)

        self.aggregate_vols = {'01-01_to_03-31':{}, '04-01_to_07-31':{}, '08-01_to_9-30':{}, '10-01_to_12-31':{}, 'Total Vol MK Test':{}}
        
        for site, stats_object in self.annual_stats_objs.items():
            # print(stats_object.site_meta_data)
            self.aggregate_meta_data[site] = stats_object.site_meta_data
            
            # self.PeakRunoffQ = self.annual_stats.runoff_timing_mk_stats.get()
            # self.PeakRunoffQ = self.annual_stats.runoff_timing_mk_stats.get()
            self.aggregate_thresholdVol[site] = stats_object.runoff_timing_mk_stats.get('Threshold Vol DOY MK Test')
            self.aggregate_thresholdDOY[site] = stats_object.runoff_timing_mk_stats.get('Threshold Vol MK Test')            

            for date_range, vol_stats in stats_object.vol_mk_stats.items():
                self.aggregate_vols.get(date_range)[site]=vol_stats
    
    def _convert_dicts_to_dfs(self):
        
        #Convert dicts to dfs
        # for site, stats_object in self.annual_stats_objs.items():
        self.aggregate_thresholdVol_df = pd.DataFrame(self.aggregate_thresholdVol).T.rename(columns={0:"Trend", 1:"h", 2:"p", 3:'z', 4:'Tau', 5:'s', 6:'var_s', 7:'slope', 8:'intercept'})
        self.aggregate_thresholdDOY_df = pd.DataFrame(self.aggregate_thresholdDOY).T.rename(columns={0:"Trend", 1:"h", 2:"p", 3:'z', 4:'Tau', 5:'s', 6:'var_s', 7:'slope', 8:'intercept'})
  
        for date_range in self.aggregate_vols.keys():
            self.aggregate_vols_dfs[date_range] = pd.DataFrame(self.aggregate_vols.get(date_range)).T.rename(columns={0:"Trend", 1:"h", 2:"p", 3:'z', 4:'Tau', 5:'s', 6:'var_s', 7:'slope', 8:'intercept'})


class aggregate_mann_kendall_plots():
    def __init__(self):
        self.figs = []

    def scatter(self, aggregate_obj_class):
        df = aggregate_obj_class.aggregate_thresholdDOY_df
        # print(df)
        fig = px.scatter(df, 'p', 'Tau', color='Trend',
                 title='Threshold DOY',
                 hover_data={
                     # 'Site Name': df.index, 
                     'Site': df.index, 
                     # 'Trend': df['Trend'],
                 },
                 symbol = df['Trend'],
                 symbol_map={'no trend':'circle-open', 'decreasing': 'triangle-down', 'increasing':'triangle-up'},
                 color_discrete_map={'decreasing':'red', 'increasing':'green', 'no trend': 'blue'},
                 opacity=0.5
                )
        self.figs.append(fig)
        fig.show()
        # fig = px.scatter(ThresholdDOY, 'p', 'tau', color='trend',
        #                  title=ThresholdDOY_title,
        #                  hover_data={
        #                      'Site Name': ThresholdDOY['site_name'], 
        #                      'Site': ThresholdDOY['sites'], 'Trend': ThresholdDOY['trend']
        #                  },
        #                  symbol = ThresholdDOY['trend'],
        #                  symbol_map={'no trend':'circle-open', 'decreasing': 'triangle-down', 'increasing':'triangle-up'},
        #                  color_discrete_map={'decreasing':'red', 'increasing':'green', 'no trend': 'blue'}
        #                 )
        # figs.append(fig)
        # fig.show()
        # with open('HeadwaterSites_MK_Results2.html', 'a') as f:
        #     f.write(fig.to_html(full_html=False, include_plotlyjs='cdn', default_width=1200, default_height=400))

        for date_range, df in aggregate_obj_class.aggregate_vols_dfs.items():
            # print(df)
            fig = px.scatter(df, 'p', 'Tau', color='Trend',
                             title=date_range,
                             hover_data={
                                 # 'Site Name': df.index, 
                                 'Site': df.index, 
                                 # 'Trend': df['Trend'],
                             },
                             symbol = df['Trend'],
                             symbol_map={'no trend':'circle-open', 'decreasing': 'triangle-down', 'increasing':'triangle-up'},
                             color_discrete_map={'decreasing':'red', 'increasing':'green', 'no trend': 'blue'},
                             opacity=0.5
                            )
            self.figs.append(fig)
            fig.show()

    def save_scatter_figs(self, file_name, delete_existing=False):
        
        if delete_existing:
            if os.path.exists(file_name):
                 os.remove(file_name)
            for fig in self.figs:       
                with open(file_name, 'a') as f:
                    f.write(fig.to_html(full_html=False, include_plotlyjs='cdn', default_width=1200, default_height=400))
        else:
            for fig in self.figs:       
                with open(file_name, 'a') as f:
                    f.write(fig.to_html(full_html=False, include_plotlyjs='cdn', default_width=1200, default_height=400))
                

In [ ]:
headwater = pd.read_excel('headwater_sites_revised.xlsx')
headwater.set_index('site_no', inplace=True)
site_list = list(headwater.index)
lats = list(headwater['lat'])
longs = list(headwater['long']) 

all_site_data = {}
for site in site_list:
    try:
        # basin = site[1][1]
        data = webservices.usgs_streamflow().get_data(start_date='1928-10-01', end_date='2023-10-01',sites=str(site)).reset_index()
        all_site_data[site] = data
    except Exception as e:
        print(e)
    finally:
        pass

In [ ]:
# start_date = '1928-10-01'
# end_date = '2018-10-01'
# start_date = pd.to_datetime(start_date).date()
# end_date = pd.to_datetime(end_date).date()
# filtered_all_site_data = {}
# for site,df in all_site_data.items():
#     filtered_df = all_site_data.get(site)[all_site_data.get(site)['Date'].between(start_date, end_date)]
#     #if more than 15% of data is missing during the time period, exclude station
#     if ((end_date - start_date).days * 0.85) > len(filtered_df['Date']):
#         print(f"Site {site} is missing more than 15% of data between {start_date} and {end_date}.  It will NOT be included in the study.")
#     else:
#         filtered_all_site_data[site] = filtered_df
#         print('Site {site} has more than 85% of data between {start_date} and {end_date} and will be included in the study.')

In [ ]:
# empty_df = 0
# df_w_data = 0
# for k,df in filtered_all_site_data.items():
#     if df.empty:
#         empty_df +=1
#     elif not df.empty:
#         df_w_data +=1
# print(empty_df)
# print(df_w_data)

In [ ]:
# yrs_of_data = []
# for site, df in all_site_data.items():
#     num_yrs_of_data = len(df['Date'])/365
#     yrs_of_data.append(num_yrs_of_data)

# print(sum(yrs_of_data)/len(yrs_of_data))
# print('--------')
# print(yrs_of_data)

In [ ]:
all_site_data.get(13073000)

In [ ]:
# start_dates = [
#     '1928-10-01', '1938-10-01', '1948-10-01', '1958-10-01', '1968-10-01', '1978-10-01', 
#     '1988-10-01', 
#     '1998-10-01', '2008-10-01', '2018-10-01'
# ] 
# end_dates = [
#     '1938-10-01', '1948-10-01', '1958-10-01', '1968-10-01', '1978-10-01', '1988-10-01', '1998-10-01', 
#     '2008-10-01', 
#     '2018-10-01', '2023-10-01'
# ] 

# '2018-10-01':'2023-10-01'
    
    # '1996-10-01':'2010-10-01'
# }



# start_date = '1988-10-01'
# end_dates = [
#     # '1968-10-01',
#     # '1978-10-01',
#     # '1988-10-01',
#     '1998-10-01', '2008-10-01', '2018-10-01', '2023-10-01']

# start_date = '1928-10-01'
# end_dates = [
#     '1938-10-01', 
#     '1948-10-01', 
#     '1958-10-01',
#     '1968-10-01',
#     '1978-10-01',
#     '1988-10-01',
#     '1998-10-01', 
#     '2008-10-01', 
#     '2018-10-01', 
#     '2023-10-01']

# start_dates = [
#     '1928-10-01',
#     # '1938-10-01', 
#     # '1948-10-01', 
#     # '1958-10-01',
#     # '1968-10-01',
#     # '1978-10-01',
#     # '1988-10-01',
#     # '1998-10-01', 
#     # '2008-10-01'
# ]
# end_date = '2018-10-01'

# start_dates = ['1989-10-01','1990-10-01','1991-10-01','1992-10-01','1993-10-01','1994-10-01','1995-10-01','1996-10-01', 
#                '1997-10-01','1998-10-01','1999-10-01','2000-10-01','2001-10-01','2002-10-01','2003-10-01', '2004-10-01',
#               '2005-10-01','2006-10-01','2007-10-01','2008-10-01','2009-10-01', '2010-10-01']
# end_date = '2018-10-01'#,]
start_date ='1959-10-01'
end_dates=['2018-10-01', '2023-10-01']

all_annual_stats_objects = {}

for end_date in end_dates:
# for start_date in start_dates:
# for start_date, end_date in zip(start_dates, end_dates):
    file_name = f'Headwater_MK_Results_{start_date}to{end_date}.html'
    # try:
    start_date = pd.to_datetime(start_date).date()
    end_date = pd.to_datetime(end_date).date()
    filtered_all_site_data = {}
    yrs_of_data = []
    for site, df in all_site_data.items():
        # num_yrs_of_data = len(df['Date'])/365
        # yrs_of_data.append(num_yrs_of_data)
    
        filtered_df = all_site_data.get(site)[all_site_data.get(site)['Date'].between(start_date, end_date)]
        #if more than 15% of data is missing during the time period, exclude station
        if ((end_date - start_date).days * 0.85) > len(filtered_df['Discharge']):
            print(f"Site {site} is missing more than 15% of data between {start_date} and {end_date}.  It will NOT be included in the study.")
        else:
            filtered_all_site_data[site] = filtered_df
            print(f'Site {site} has more than 85% of data between {start_date} and {end_date} and will be included in the study.')
        
     
    
    
        
    annual_stats_objects = {}
    # annual_stats_df = {}
    for site, df in filtered_all_site_data.items():
    #     # try:
    #     # print(site)
    #     if df.empty:
    #         print(f'{site} has no data for the period between {start_date} and {end_date}.  Moving to next site.')
    #     else:
    #         print(f'Filtering data for {site} between {start_date} and {end_date}')
            d = dataloader.DataLoader(df, 'Date', name_of_Q_column='Discharge')
            # print(d._df.head(1))
            s = climatestatistics.Streamflow(d)
            start_date = pd.to_datetime(start_date)
            end_date = pd.to_datetime(end_date)
            site_results = annual_stats(s)
        
            site_results.calc_stats(start_date, end_date)
            annual_stats_objects[site] = site_results
            # print(site_results)
            # annual_stats_df[site] = site_results.aggregate_vols_dfs


    # print(f'number of sites included in study: {len(annual_stats_objects)}')
    all_annual_stats_objects[f'{start_date}to{end_date}'] = annual_stats_objects
            
        # except Exception as e:
        #     print('Failed importing data or running climate statistics.')
        #     print(e)
        #     traceback_str = ''.join(traceback.format_tb(e.__traceback__))
        #     print(traceback_str)
    print(annual_stats_objects)
    print(len(annual_stats_objects))
    ammassed_results = aggregate_stats(annual_stats_objs=annual_stats_objects)
    ammassed_results._aggregate_stats()
    ammassed_results._convert_dicts_to_dfs()

    agg_plots = aggregate_mann_kendall_plots()
    agg_plots.scatter(ammassed_results)
    agg_plots.save_scatter_figs(file_name=file_name, delete_existing=True)

    # return ammassed_results

In [ ]:
# all_annual_stats_objects.get('1988-10-01 00:00:00to2008-10-01 00:00:00').get(14097100).vol_mk_stats
all_annual_stats_objects.get('1988-10-01 00:00:00to2008-10-01 00:00:00').get(14097100).runoff_timing_mk_stats

In [ ]:
for date_range, df in ammassed_results.aggregate_vols_dfs.items():
    print(list(df.index))
    print(len(list(df.index)))

In [ ]:
# ammassed_results.aggregate_vols_dfs

In [ ]:
ammassed_results.aggregate_thresholdDOY_df.index

In [ ]:
len(ammassed_results.aggregate_thresholdDOY_df.index)

In [ ]:
# date_ranges = {'01-01_to_03-31':{}, '04-01_to_06-30':{}, '07-01_to_9-30':{}, '10-01_to_12-31':{}, 'Total Vol MK Test':{}}
# for dr in date_ranges.keys():
#     df = pd.DataFrame(ammassed_results.aggregate_vols.get(dr)).T.rename(columns={0:"Trend", 1:"h", 2:"p", 3:'z', 4:'Tau', 5:'s', 6:'var_s', 7:'slope', 8:'intercept'})
#     print(df)

In [ ]:
figs = []
# fig = px.scatter(peak_runoffQ, 'p', 'tau', color='trend',
#                  title=peak_runoffQ_title,
#                  hover_data={
#                      'Site Name': peak_runoffQ['site_name'], 
#                      'Site': peak_runoffQ['sites'], 'Trend': peak_runoffQ['trend'],},
#                  symbol = peak_runoffQ['trend'],
#                  symbol_map={'no trend':'circle-open', 'decreasing': 'triangle-down', 'increasing':'triangle-up'},
#                  color_discrete_map={'decreasing':'red', 'increasing':'green', 'no trend': 'blue'}

#                 )
# figs.append(fig)
# fig.show()
# 
# fig = px.scatter(peak_runoffDOY, 'p', 'tau', color='trend',
#                  title=peak_runoffDOY_title,
#                  hover_data={
#                      'Site Name': peak_runoffDOY['site_name'], 
#                      'Site': peak_runoffDOY['sites'], 'Trend': peak_runoffDOY['trend'],},
#                  symbol = peak_runoffDOY['trend'],
#                  symbol_map={'no trend':'circle-open', 'decreasing': 'triangle-down', 'increasing':'triangle-up'},
#                  color_discrete_map={'decreasing':'red', 'increasing':'green', 'no trend': 'blue'}

#                 )
# figs.append(fig)
# fig.show()


fig = px.scatter(ThresholdVol, 'p', 'tau', color='trend',
                 title=ThresholdVol_title,
                 hover_data={
                     'Site Name': ThresholdVol['site_name'], 
                     'Site': ThresholdVol['sites'], 'Trend': ThresholdVol['trend'],},
                 symbol = ThresholdVol['trend'],
                 symbol_map={'no trend':'circle-open', 'decreasing': 'triangle-down', 'increasing':'triangle-up'},
                 color_discrete_map={'decreasing':'red', 'increasing':'green', 'no trend': 'blue'}
                )
figs.append(fig)
fig.show()
fig = px.scatter(ThresholdDOY, 'p', 'tau', color='trend',
                 title=ThresholdDOY_title,
                 hover_data={
                     'Site Name': ThresholdDOY['site_name'], 
                     'Site': ThresholdDOY['sites'], 'Trend': ThresholdDOY['trend'],},
                 symbol = ThresholdDOY['trend'],
                 symbol_map={'no trend':'circle-open', 'decreasing': 'triangle-down', 'increasing':'triangle-up'},
                 color_discrete_map={'decreasing':'red', 'increasing':'green', 'no trend': 'blue'}
                )
figs.append(fig)
fig.show()
# with open('HeadwaterSites_MK_Results2.html', 'a') as f:
#     f.write(fig.to_html(full_html=False, include_plotlyjs='cdn', default_width=1200, default_height=400))

fig = px.scatter(total_vol, 'p', 'tau', color='trend',
                 title=total_vol_title,
                 hover_data={
                     'Site Name': total_vol['site_name'], 
                     'Site': total_vol['sites'], 'Trend': total_vol['trend'],},
                 symbol = total_vol['trend'],
                 symbol_map={'no trend':'circle-open', 'decreasing': 'triangle-down', 'increasing':'triangle-up'},
                 color_discrete_map={'decreasing':'red', 'increasing':'green', 'no trend': 'blue'}
                )
figs.append(fig)
fig.show()

fig = px.scatter(SummerVol, 'p', 'tau', color='trend',
                 title=SummerVol_title,
                 hover_data={
                     'Site Name': SummerVol['site_name'], 
                     'Site': SummerVol['sites'], 'Trend': SummerVol['trend'],},
                 symbol = SummerVol['trend'],
                 symbol_map={'no trend':'circle-open', 'decreasing': 'triangle-down', 'increasing':'triangle-up'},
                 color_discrete_map={'decreasing':'red', 'increasing':'green', 'no trend': 'blue'}
                )
figs.append(fig)
fig.show()

fig = px.scatter(FallVol, 'p', 'tau', color='trend',
                 title=FallVol_title,
                 hover_data={
                     'Site Name': FallVol['site_name'], 
                     'Site': FallVol['sites'], 'Trend': FallVol['trend'],},
                 symbol = FallVol['trend'],
                 symbol_map={'no trend':'circle-open', 'decreasing': 'triangle-down', 'increasing':'triangle-up'},
                 color_discrete_map={'decreasing':'red', 'increasing':'green', 'no trend': 'blue'}
                )
figs.append(fig)
fig.show()

fig = px.scatter(WinterVol, 'p', 'tau', color='trend',
                 title=WinterVol_title,
                 hover_data={
                     'Site Name': WinterVol['site_name'], 
                     'Site': WinterVol['sites'], 'Trend': WinterVol['trend'],},
                 symbol = WinterVol['trend'],
                 symbol_map={'no trend':'circle-open', 'decreasing': 'triangle-down', 'increasing':'triangle-up'},
                 color_discrete_map={'decreasing':'red', 'increasing':'green', 'no trend': 'blue'}
                )
figs.append(fig)
fig.show()

fig = px.scatter(RunoffVol, 'p', 'tau', color='trend',
                 title=RunoffVol_title,
                 hover_data={
                     'Site Name': RunoffVol['site_name'], 
                     'Site': RunoffVol['sites'], 'Trend': RunoffVol['trend'],},
                 symbol = RunoffVol['trend'],
                 symbol_map={'no trend':'circle-open', 'decreasing': 'triangle-down', 'increasing':'triangle-up'},
                 color_discrete_map={'decreasing':'red', 'increasing':'green', 'no trend': 'blue'}
                )
figs.append(fig)
fig.show()

file_name = f'Headwater_MK_Results_{start_date[:4]}to{end_date[:4]}.html'
if os.path.exists(file_name):
     os.remove(file_name)
for fig in figs:       
    with open(file_name, 'a') as f:
        f.write(fig.to_html(full_html=False, include_plotlyjs='cdn', default_width=1200, default_height=400))


In [ ]:
         
        
    # sites = []
    # site_name = []
    # lat = []
    # long = []
    # # basins = []
    
    # PeakRunoffQ_trend = []
    # PeakRunoffQ_h = []
    # PeakRunoffQ_p = []
    # PeakRunoffQ_tau = []
    
    # PeakRunoffDOY_trend = []
    # PeakRunoffDOY_h = []
    # PeakRunoffDOY_p = []
    # PeakRunoffDOY_tau = []
    
    # ThresholdVol_trend = []
    # ThresholdVol_h = []
    # ThresholdVol_p = []
    # ThresholdVol_tau = []
    
    # ThresholdDOY_trend = []
    # ThresholdDOY_p = []
    # ThresholdDOY_tau = []
    
    # TotalVol_trend = []
    # TotalVol_h = []
    # TotalVol_p = []
    # TotalVol_tau = []
    
    # RunoffVol_trend = []
    # RunoffVol_h = []
    # RunoffVol_p = []
    # RunoffVol_tau = []
    
    # SummerVol_trend = []
    # SummerVol_h = []
    # SummerVol_p = []
    # SummerVol_tau = []
    
    # FallVol_trend = []
    # FallVol_h = []
    # FallVol_p = []
    # FallVol_tau = []
    
    # WinterVol_trend = []
    # WinterVol_h = []
    # WinterVol_p = []
    # WinterVol_tau = []
    
    # for site, value in volume_stats_for_vip_sites.items():
        
    #     sites.append(site)
    #     site_name.append(value.get('name'))
    #     lat.append(value.get('lat'))
    #     long.append(value.get('long'))              
                         
    #     # basins.append(site[1]['Basin'])
        
    #     PeakRunoffQ_trend.append(value.get('Q on date of peak Q date MK Test')[0])
    #     PeakRunoffQ_p.append(value.get('Q on date of peak Q date MK Test')[2])
    #     PeakRunoffQ_tau.append(value.get('Q on date of peak Q date MK Test')[4])
        
    #     PeakRunoffDOY_trend.append(value.get('Q on date of peak Q date MK Test')[0])
    #     PeakRunoffDOY_p.append(value.get('Q on date of peak Q date MK Test')[2])
    #     PeakRunoffDOY_tau.append(value.get('Q on date of peak Q date MK Test')[4])    
            
    #     ThresholdVol_trend.append(value.get('Threshold Vol MK Test')[0])
    #     ThresholdVol_p.append(value.get('Threshold Vol MK Test')[2])
    #     ThresholdVol_tau.append(value.get('Threshold Vol MK Test')[4])    
            
    #     ThresholdDOY_trend.append(value.get('Threshold Vol DOY MK Test')[0])
    #     ThresholdDOY_p.append(value.get('Threshold Vol DOY MK Test')[2])
    #     ThresholdDOY_tau.append(value.get('Threshold Vol DOY MK Test')[4])            
            
    #     TotalVol_trend.append(value.get('Total Vol MK Test')[0])
    #     TotalVol_p.append(value.get('Total Vol MK Test')[2])
    #     TotalVol_tau.append(value.get('Total Vol MK Test')[4])
        
    #     SummerVol_trend.append(value.get('Summer Vol')[0])
    #     SummerVol_p.append(value.get('Summer Vol')[2])
    #     SummerVol_tau.append(value.get('Summer Vol')[4])
    
    #     RunoffVol_trend.append(value.get('Runoff Season Vol')[0])
    #     RunoffVol_p.append(value.get('Runoff Season Vol')[2])
    #     RunoffVol_tau.append(value.get('Runoff Season Vol')[4])
        
    #     WinterVol_trend.append(value.get('Winter Vol')[0])
    #     WinterVol_p.append(value.get('Winter Vol')[2])
    #     WinterVol_tau.append(value.get('Winter Vol')[4])
    
    #     FallVol_trend.append(value.get('Fall Vol')[0])
    #     FallVol_p.append(value.get('Fall Vol')[2])
    #     FallVol_tau.append(value.get('Fall Vol')[4])
    
    
    
    # peak_runoffQ = pd.DataFrame({'sites':sites, 'site_name': site_name, 'lat': lat, 'long': long, 'trend': PeakRunoffQ_trend,  'p': PeakRunoffQ_p, 'tau': PeakRunoffQ_tau})
    # peak_runoffDOY = pd.DataFrame({'sites':sites, 'site_name': site_name, 'lat': lat, 'long': long, 'trend': PeakRunoffDOY_trend,  'p': PeakRunoffDOY_p, 'tau': PeakRunoffDOY_tau})
    # ThresholdVol = pd.DataFrame({'sites':sites, 'site_name': site_name, 'lat': lat, 'long': long, 'trend': ThresholdVol_trend, 'p': ThresholdVol_p, 'tau': ThresholdVol_tau})
    # ThresholdDOY = pd.DataFrame({'sites':sites, 'site_name': site_name, 'lat': lat, 'long': long, 'trend': ThresholdDOY_trend, 'p': ThresholdDOY_p, 'tau': ThresholdDOY_tau})
    # total_vol = pd.DataFrame({'sites':sites, 'site_name': site_name, 'lat': lat, 'long': long, 'trend': TotalVol_trend, 'p': TotalVol_p, 'tau': TotalVol_tau})
    # RunoffVol = pd.DataFrame({'sites':sites, 'site_name': site_name, 'lat': lat, 'long': long, 'trend': RunoffVol_trend, 'p': RunoffVol_p, 'tau': RunoffVol_tau})
    # SummerVol = pd.DataFrame({'sites':sites, 'site_name': site_name, 'lat': lat, 'long': long, 'trend': SummerVol_trend, 'p': SummerVol_p, 'tau': SummerVol_tau})
    # WinterVol = pd.DataFrame({'sites':sites, 'site_name': site_name, 'lat': lat, 'long': long, 'trend': WinterVol_trend, 'p': WinterVol_p, 'tau': WinterVol_tau})
    # FallVol = pd.DataFrame({'sites':sites, 'site_name': site_name, 'lat': lat, 'long': long, 'trend': FallVol_trend, 'p': FallVol_p, 'tau': FallVol_tau})
    
    # peak_runoffQ_title = f'Trend in Peak Runoff Volume at Headwater Sites: MK Results {start_date[:4]} to {end_date[:4]}' 
    # peak_runoffDOY_title = f'Trend in Timing During Peak at Headwater Sites: MK Results {start_date[:4]} to {end_date[:4]}'
    # ThresholdDOY_title = f'Trends in Timing of 50% Yearly Volume at Headwater Sites: MK Results {start_date[:4]} to {end_date[:4]}'
    # ThresholdVol_title = f'Trend in Volume at 50% Yearly Flow at Headwater Sites: MK Results {start_date[:4]} to {end_date[:4]}'
    # total_vol_title = f'Yearly Total Volume Trends at Sites at Headwater Sites: MK Results {start_date[:4]} to {end_date[:4]}'
    # SummerVol_title = f'Trend in Late Summer Volumes (08-01 through 09-30) at Headwater Sites: MK Results {start_date[:4]} to {end_date[:4]}'
    # FallVol_title = f'Trend in Fall to early Winter Volumes (10-01 through 12-31) at Headwater Sites: MK Results {start_date[:4]} to {end_date[:4]}'
    # WinterVol_title = f'Trend in Wintertime Volumes (01-01 through 4-01)  at Headwater Sites: MK Results {start_date[:4]} to {end_date[:4]}'
    # RunoffVol_title = f'Trend in Runoff Season Volumes (04-01 through 07-31) at Headwater Sites: MK Results {start_date[:4]} to {end_date[:4]}'
    
    
    # figs = []
    # fig = px.scatter(peak_runoffQ, 'p', 'tau', color='trend',
    #                  title=peak_runoffQ_title,
    #                  hover_data={
    #                      'Site Name': peak_runoffQ['site_name'], 
    #                      'Site': peak_runoffQ['sites'], 'Trend': peak_runoffQ['trend'],},
    #                  symbol = peak_runoffQ['trend'],
    #                  symbol_map={'no trend':'circle-open', 'decreasing': 'triangle-down', 'increasing':'triangle-up'},
    #                  color_discrete_map={'decreasing':'red', 'increasing':'green', 'no trend': 'blue'}
    
    #                 )
    # figs.append(fig)
    # fig.show()
    
    # fig = px.scatter(peak_runoffDOY, 'p', 'tau', color='trend',
    #                  title=peak_runoffDOY_title,
    #                  hover_data={
    #                      'Site Name': peak_runoffDOY['site_name'], 
    #                      'Site': peak_runoffDOY['sites'], 'Trend': peak_runoffDOY['trend'],},
    #                  symbol = peak_runoffDOY['trend'],
    #                  symbol_map={'no trend':'circle-open', 'decreasing': 'triangle-down', 'increasing':'triangle-up'},
    #                  color_discrete_map={'decreasing':'red', 'increasing':'green', 'no trend': 'blue'}
    
    #                 )
    # figs.append(fig)
    # fig.show()
    
    
    # fig = px.scatter(ThresholdVol, 'p', 'tau', color='trend',
    #                  title=ThresholdVol_title,
    #                  hover_data={
    #                      'Site Name': ThresholdVol['site_name'], 
    #                      'Site': ThresholdVol['sites'], 'Trend': ThresholdVol['trend'],},
    #                  symbol = ThresholdVol['trend'],
    #                  symbol_map={'no trend':'circle-open', 'decreasing': 'triangle-down', 'increasing':'triangle-up'},
    #                  color_discrete_map={'decreasing':'red', 'increasing':'green', 'no trend': 'blue'}
    #                 )
    # figs.append(fig)
    # fig.show()
    # fig = px.scatter(ThresholdDOY, 'p', 'tau', color='trend',
    #                  title=ThresholdDOY_title,
    #                  hover_data={
    #                      'Site Name': ThresholdDOY['site_name'], 
    #                      'Site': ThresholdDOY['sites'], 'Trend': ThresholdDOY['trend'],},
    #                  symbol = ThresholdDOY['trend'],
    #                  symbol_map={'no trend':'circle-open', 'decreasing': 'triangle-down', 'increasing':'triangle-up'},
    #                  color_discrete_map={'decreasing':'red', 'increasing':'green', 'no trend': 'blue'}
    #                 )
    # figs.append(fig)
    # fig.show()
    # # with open('HeadwaterSites_MK_Results2.html', 'a') as f:
    # #     f.write(fig.to_html(full_html=False, include_plotlyjs='cdn', default_width=1200, default_height=400))
    
    # fig = px.scatter(total_vol, 'p', 'tau', color='trend',
    #                  title=total_vol_title,
    #                  hover_data={
    #                      'Site Name': total_vol['site_name'], 
    #                      'Site': total_vol['sites'], 'Trend': total_vol['trend'],},
    #                  symbol = total_vol['trend'],
    #                  symbol_map={'no trend':'circle-open', 'decreasing': 'triangle-down', 'increasing':'triangle-up'},
    #                  color_discrete_map={'decreasing':'red', 'increasing':'green', 'no trend': 'blue'}
    #                 )
    # figs.append(fig)
    # fig.show()
    
    # fig = px.scatter(SummerVol, 'p', 'tau', color='trend',
    #                  title=SummerVol_title,
    #                  hover_data={
    #                      'Site Name': SummerVol['site_name'], 
    #                      'Site': SummerVol['sites'], 'Trend': SummerVol['trend'],},
    #                  symbol = SummerVol['trend'],
    #                  symbol_map={'no trend':'circle-open', 'decreasing': 'triangle-down', 'increasing':'triangle-up'},
    #                  color_discrete_map={'decreasing':'red', 'increasing':'green', 'no trend': 'blue'}
    #                 )
    # figs.append(fig)
    # fig.show()
    
    # fig = px.scatter(FallVol, 'p', 'tau', color='trend',
    #                  title=FallVol_title,
    #                  hover_data={
    #                      'Site Name': FallVol['site_name'], 
    #                      'Site': FallVol['sites'], 'Trend': FallVol['trend'],},
    #                  symbol = FallVol['trend'],
    #                  symbol_map={'no trend':'circle-open', 'decreasing': 'triangle-down', 'increasing':'triangle-up'},
    #                  color_discrete_map={'decreasing':'red', 'increasing':'green', 'no trend': 'blue'}
    #                 )
    # figs.append(fig)
    # fig.show()
    
    # fig = px.scatter(WinterVol, 'p', 'tau', color='trend',
    #                  title=WinterVol_title,
    #                  hover_data={
    #                      'Site Name': WinterVol['site_name'], 
    #                      'Site': WinterVol['sites'], 'Trend': WinterVol['trend'],},
    #                  symbol = WinterVol['trend'],
    #                  symbol_map={'no trend':'circle-open', 'decreasing': 'triangle-down', 'increasing':'triangle-up'},
    #                  color_discrete_map={'decreasing':'red', 'increasing':'green', 'no trend': 'blue'}
    #                 )
    # figs.append(fig)
    # fig.show()
    
    # fig = px.scatter(RunoffVol, 'p', 'tau', color='trend',
    #                  title=RunoffVol_title,
    #                  hover_data={
    #                      'Site Name': RunoffVol['site_name'], 
    #                      'Site': RunoffVol['sites'], 'Trend': RunoffVol['trend'],},
    #                  symbol = RunoffVol['trend'],
    #                  symbol_map={'no trend':'circle-open', 'decreasing': 'triangle-down', 'increasing':'triangle-up'},
    #                  color_discrete_map={'decreasing':'red', 'increasing':'green', 'no trend': 'blue'}
    #                 )
    # figs.append(fig)
    # fig.show()
    
    # file_name = f'Headwater_MK_Results_{start_date[:4]}to{end_date[:4]}.html'
    # if os.path.exists(file_name):
    #      os.remove(file_name)
    # for fig in figs:       
    #     with open(file_name, 'a') as f:
    #         f.write(fig.to_html(full_html=False, include_plotlyjs='cdn', default_width=1200, default_height=400))


In [ ]:


# for site in site_list:
#     try:
#         # basin = site[1][1]
#         data = webservices.usgs_streamflow().get_data(start_date='1970-10-01', end_date='2018-10-01',sites=str(site)).reset_index()
#         d = dataloader.DataLoader(data, 'Date', name_of_Q_column='Discharge')
#         s = climatestatistics.Streamflow(d)
#     except Exception as e:
#         print(e)
#     finally:
#         pass
    
#     try:
#         mann_kendall_results = {}

#         s.calc_max(calc_from_rolling_median=False, window_size=7)
#         print(f'Mann-Kendall Test Results looking for trend in day of year on peak runoff dates: {s.rolling_yr_Qmax_mk_test}')
#         # print(f'Mann-Kendall Test Results looking for trend in discharge on peak runoff dates: {s.rolling_yr_DOYmax_mk_test}')
#     #     peak_runoff_plot = streamflow_stat_plots.Plot_Peak_Runoff(s)
#     #     peak_runoff_plot = peak_runoff_plot.plot_peak_runoff(site[0])
#         print('-------------------------------------------------')  
    
#         s.calc_annual_runoff_threshold_day(0.5, alpha=.05)
#     #     volume_stats_for_vip_sites[site[1]] = s.threshold_vol_stats 
#         # print(f'Mann-Kendall Test Results looking for trend in {s._percent*100}% runoff timing: {s.threshold_vol_mann_kendall_test}')
#         # print(f'Mann-Kendall Test Results looking for trend in total annual discharge: {s.total_volume_mann_kendall_test}')  
#     #     threshold_runoff_plot = streamflow_stat_plots.Plot_Runoff_Threshold(s)
#     #     threshold_runoff_plot.plot_runoff_threshold(site[0])
    
#     #     threshold_runoff_plot = streamflow_stat_plots_matplotlib.Plot_Runoff_Threshold(s)
#     #     threshold_runoff_plot.plot_runoff_threshold(site[0])
#         # print('-------------------------------------------------')
        
#     #     for day in ['01-01', '03-01', '05-01', '08-01']:
#         s.calc_runoff_bw_days(begin_month_day='01-01', end_month_day='12-31', alpha=.05)
#         # print(f'Mann-Kendall Test Results looking for trend in discharge from {s.begin_month_day} to {s.end_month_day}: {s.volume_bw_days_mann_kendall_test}')
#     #     runoff_bw_dates_plot = streamflow_stat_plots.Plot_Runoff_Volume_Between_2Days(s)
#     #     runoff_bw_dates_plot.plot_runoff_volume_between_2days(site[0])
    
#         # print('-------------------------------------------------')      
# #         dates_ranges = {'01-01':'04-01', '04-01':'07-31', '08-01':'09-30', '10-01':'12-31'} 

#         s.calc_runoff_bw_days(begin_month_day='10-01', end_month_day='12-31', alpha=.05)
#         # print(f'Mann-Kendall Test Results looking for trend in discharge from {s.begin_month_day} to {s.end_month_day}: {s.volume_bw_days_mann_kendall_test}')
#         runoff_bw_dates_plot = streamflow_stat_plots.Plot_Runoff_Volume_Between_2Days(s)
#         # runoff_bw_dates_plot.plot_runoff_volume_between_2days(site[0])
#         mann_kendall_results['Fall Vol']=s.volume_bw_days_mann_kendall_test
#         # print('-------------------------------------------------')
#     #     for day in ['01-01', '03-01', '05-01', '08-01']:
#         s.calc_runoff_bw_days(begin_month_day='01-01', end_month_day='04-01', alpha=.05)
#         # print(f'Mann-Kendall Test Results looking for trend in discharge from {s.begin_month_day} to {s.end_month_day}: {s.volume_bw_days_mann_kendall_test}')
#         runoff_bw_dates_plot = streamflow_stat_plots.Plot_Runoff_Volume_Between_2Days(s)
#         # runoff_bw_dates_plot.plot_runoff_volume_between_2days(site[0])
#         mann_kendall_results['Winter Vol']=s.volume_bw_days_mann_kendall_test
#         # print('-------------------------------------------------')
#     #     for day in ['01-01', '03-01', '05-01', '08-01']:
#         s.calc_runoff_bw_days(begin_month_day='04-01', end_month_day='07-01', alpha=.05)
#         # print(f'Mann-Kendall Test Results looking for trend in discharge from {s.begin_month_day} to {s.end_month_day}: {s.volume_bw_days_mann_kendall_test}')
#     #     runoff_bw_dates_plot2 = streamflow_stat_plots.Plot_Runoff_Volume_Between_2Days(s)
#     #     runoff_bw_dates_plot2.plot_runoff_volume_between_2days(site[0])
#         mann_kendall_results['Runoff Season Vol'] = s.volume_bw_days_mann_kendall_test
#         # print('-------------------------------------------------')            
#     #     for day in ['01-01', '03-01', '05-01', '08-01']:
#         s.calc_runoff_bw_days(begin_month_day='07-01', end_month_day='9-30', alpha=.05)
#         # print(f'Mann-Kendall Test Results looking for trend in discharge from {s.begin_month_day} to {s.end_month_day}: {s.volume_bw_days_mann_kendall_test}')
#     #     runoff_bw_dates_plot3 = streamflow_stat_plots.Plot_Runoff_Volume_Between_2Days(s)
#         mann_kendall_results['Summer Vol'] = s.volume_bw_days_mann_kendall_test

#     #     runoff_bw_dates_plot3.plot_runoff_volume_between_2days(site[0])
       
        
#         mann_kendall_results['Q on date of peak Q date MK Test'] = s.rolling_yr_Qmax_mk_test
#         mann_kendall_results['dayofyear on date of peak Q date MK Test'] = s.rolling_yr_DOYmax_mk_test
#         mann_kendall_results['Threshold Vol MK Test'] = s.threshold_vol_mann_kendall_test
#         mann_kendall_results['Threshold Vol DOY MK Test'] = s.threshold_vol_dates_mann_kendall_test
#         mann_kendall_results['Total Vol MK Test'] = s.total_volume_mann_kendall_test
#         # mann_kendall_results['Basin'] = basin
#         mann_kendall_results['lat'] = headwater[headwater.index==site].iat[0,1]
#         mann_kendall_results['long'] = headwater[headwater.index==site].iat[0,2]
#         mann_kendall_results['name'] = headwater[headwater.index==site].iat[0,0]
#         volume_stats_for_vip_sites[site] = mann_kendall_results

#         # print(mann_kendall_results)
#     except Exception as e:
#         print(e)
#     finally:
#         pass
# print('---------------------------------------------------------------------------------------------------')    

# Finegling 

In [ ]:
# site_list

In [ ]:
import pandas as pd
import requests
import os
import glob
import datetime as datetime
import pandas as pd
from dataIO import dataloader, webservices
from statisticscalculator import generalstatistics, climatestatistics
from plot_collection import stackedlineplots, streamflow_stat_plots, streamflow_stat_plots_matplotlib
import plotly.express as px
import plotly.graph_objects as go

In [ ]:
headwater = pd.read_excel('headwater_sites_revised.xlsx')
headwater.set_index('site_no', inplace=True)
site_list = list(headwater.index)
lats = list(headwater['lat'])
longs = list(headwater['long']) 


In [ ]:

all_site_data = {}
for site in site_list:
    try:
        # basin = site[1][1]
        data = webservices.usgs_streamflow().get_data(start_date='1928-10-01', end_date='2023-10-01',sites=str(site)).reset_index()
        all_site_data[site] = data
    except Exception as e:
        print(e)
    finally:
        pass

In [ ]:
# all_site_data

In [ ]:
# end_date = '2018-10-01'
# start_dates = [
#     # '1938-10-01', 
#     '1948-10-01', 
#     '1968-10-01',
#     '1975-10-01',
#     # '1978-10-01',
#     # '1988-10-01',
#     # '1998-10-01', 
# #     '2010-10-01', 
# #     '2018-10-01', 
# ]
# # dates = ['1998-10-01']#, '1948-10-01', '1958-10-01','1968-10-01','1978-10-01', '1988-10-01', '1998-10-01', '2008-10-01']#, '2018-10-01']

# all_annual_stats_objects = {}

# # for end_date in end_dates:
# for start_date in start_dates:
# # for start_date, end_date in zip(start_dates, end_dates):
#     file_name = f'Headwater_MK_Results_{start_date}to{end_date}.html'
#     # try:
#     start_date = pd.to_datetime(start_date).date()
#     end_date = pd.to_datetime(end_date).date()
#     filtered_all_site_data = {}
#     yrs_of_data = []
#     for site, df in all_site_data.items():
#         # num_yrs_of_data = len(df['Date'])/365
#         # yrs_of_data.append(num_yrs_of_data)
    
#         filtered_df = all_site_data.get(site)[all_site_data.get(site)['Date'].between(start_date, end_date)]
#         #if more than 15% of data is missing during the time period, exclude station
#         if ((end_date - start_date).days * 0.85) > len(filtered_df['Discharge']):
#             print(f"Site {site} is missing more than 15% of data between {start_date} and {end_date}.  It will NOT be included in the study.")
#         else:
#             filtered_all_site_data[site] = filtered_df
#             print(f'Site {site} has more than 85% of data between {start_date} and {end_date} and will be included in the study.')
 

In [ ]:
filtered_all_site_data.get(12304500)

In [ ]:
end_date = '2018-10-01'
start_dates = [
    # '1938-10-01', 
    '1948-10-01', 
    '1968-10-01',
    '1975-10-01',
    # '1978-10-01',
    # '1988-10-01',
    # '1998-10-01', 
#     '2010-10-01', 
#     '2018-10-01', 
]
# dates = ['1998-10-01']#, '1948-10-01', '1958-10-01','1968-10-01','1978-10-01', '1988-10-01', '1998-10-01', '2008-10-01']#, '2018-10-01']
# for start_date in start_dates:
    
#     file_name = f'MF_Flow_MK_Results_{start_date[:4]}to{end_date[:4]}.html'
    
#     start_date = start_date
#     # end_date = end_date
#     modified_flows_m_filtered_dates = {}
#     for k,v in modified_flows_m.items():
#         # modified_flows_m_filtered_dates[k] = modified_flows_m.get(k)[modified_flows_m.get(k)['date']>=start_date]
#         modified_flows_m_filtered_dates[k] = modified_flows_m.get(k)[modified_flows_m.get(k)['date'].between(start_date, end_date)]
    
for start_date in start_dates:
# for start_date, end_date in zip(start_dates, end_dates):
    file_name = f'Headwater_MK_Results_{start_date}to{end_date}.html'
    # try:
    start_date = pd.to_datetime(start_date).date()
    end_date = pd.to_datetime(end_date).date()
    filtered_all_site_data = {}
    yrs_of_data = []
    for site, df in all_site_data.items():
        # num_yrs_of_data = len(df['Date'])/365
        # yrs_of_data.append(num_yrs_of_data)
    
        filtered_df = all_site_data.get(site)[all_site_data.get(site)['Date'].between(start_date, end_date)]
        #if more than 15% of data is missing during the time period, exclude station
        if ((end_date - start_date).days * 0.85) > len(filtered_df['Discharge']):
            print(f"Site {site} is missing more than 15% of data between {start_date} and {end_date}.  It will NOT be included in the study.")
        else:
            filtered_all_site_data[site] = filtered_df
            print(f'Site {site} has more than 85% of data between {start_date} and {end_date} and will be included in the study.')
     
    
    volume_stats_for_vip_sites = {}
    
    for site in filtered_all_site_data.items():
        d = dataloader.DataLoader(site[1], 'Date', name_of_Q_column='Discharge')
        s = climatestatistics.Streamflow(d)

    
    # print('-------------------------------------------------')  
#         try:
#             mann_kendall_results = {}
         
#             s.calc_max(calc_from_rolling_median=False, window_size=7)  
#             s.calc_annual_runoff_threshold_day(0.5, alpha=.05)
#             s.calc_runoff_bw_days(begin_month_day='01-01', end_month_day='12-31', alpha=.05)
#             s.calc_runoff_bw_days(begin_month_day='10-01', end_month_day='12-31', alpha=.05)
            
#             runoff_bw_dates_plot = streamflow_stat_plots.Plot_Runoff_Volume_Between_2Days(s)
#             mann_kendall_results['Fall Vol']=s.volume_bw_days_mann_kendall_test

#             s.calc_runoff_bw_days(begin_month_day='01-01', end_month_day='03-31', alpha=.05)
#             runoff_bw_dates_plot = streamflow_stat_plots.Plot_Runoff_Volume_Between_2Days(s)
#             mann_kendall_results['Winter Vol']=s.volume_bw_days_mann_kendall_test

#             s.calc_runoff_bw_days(begin_month_day='04-01', end_month_day='06-30', alpha=.05)

#             mann_kendall_results['Runoff Season Vol'] = s.volume_bw_days_mann_kendall_test

#             s.calc_runoff_bw_days(begin_month_day='07-01', end_month_day='9-30', alpha=.05)

#             mann_kendall_results['Summer Vol'] = s.volume_bw_days_mann_kendall_test
           
            
#             mann_kendall_results['Q on date of peak Q date MK Test'] = s.rolling_yr_Qmax_mk_test
#             mann_kendall_results['dayofyear on date of peak Q date MK Test'] = s.rolling_yr_DOYmax_mk_test
#             mann_kendall_results['Threshold Vol MK Test'] = s.threshold_vol_mann_kendall_test
#             mann_kendall_results['Threshold Vol DOY MK Test'] = s.threshold_vol_dates_mann_kendall_test
#             mann_kendall_results['Total Vol MK Test'] = s.total_volume_mann_kendall_test
#             # mann_kendall_results['Basin'] = basin
            
            
#             mann_kendall_results['lat'] = headwater[headwater.index==site].iat[0,1]
#             mann_kendall_results['long'] = headwater[headwater.index==site].iat[0,2]
#             mann_kendall_results['name'] = headwater[headwater.index==site].iat[0,0]
#             mann_kendall_results['site'] = site
            
#             volume_stats_for_vip_sites[site_short] = mann_kendall_results
            
#         except Exception as e:
#             raise ValueError('Saving to volume_stats_for_vip_sites prob failed')
#             print(e)
    try:
        mann_kendall_results = {}
        mann_kendall_results['lat'] = headwater[headwater.index==site].iat[0,1]
        mann_kendall_results['long'] = headwater[headwater.index==site].iat[0,2]
        mann_kendall_results['name'] = headwater[headwater.index==site].iat[0,0]
        s.calc_max(calc_from_rolling_median=False, window_size=7)
        s.calc_annual_runoff_threshold_day(0.5, alpha=.05)
        s.calc_runoff_bw_days(begin_month_day='01-01', end_month_day='12-31', alpha=.05)
        s.calc_runoff_bw_days(begin_month_day='10-01', end_month_day='12-31', alpha=.05)
        # print(f'Mann-Kendall Test Results looking for trend in discharge from {s.begin_month_day} to {s.end_month_day}: {s.volume_bw_days_mann_kendall_test}')
        runoff_bw_dates_plot = streamflow_stat_plots.Plot_Runoff_Volume_Between_2Days(s)
        # runoff_bw_dates_plot.plot_runoff_volume_between_2days(site[0])
        mann_kendall_results['Fall Vol']=s.volume_bw_days_mann_kendall_test
        # print('-------------------------------------------------')
    #     for day in ['01-01', '03-01', '05-01', '08-01']:
        s.calc_runoff_bw_days(begin_month_day='01-01', end_month_day='03-31', alpha=.05)
        print(f'Mann-Kendall Test Results looking for trend in discharge from {s.begin_month_day} to {s.end_month_day}: {s.volume_bw_days_mann_kendall_test}')
        runoff_bw_dates_plot = streamflow_stat_plots.Plot_Runoff_Volume_Between_2Days(s)
        # runoff_bw_dates_plot.plot_runoff_volume_between_2days(site[0])
        mann_kendall_results['Winter Vol']=s.volume_bw_days_mann_kendall_test
        print('-------------------------------------------------')
    #     for day in ['01-01', '03-01', '05-01', '08-01']:
        s.calc_runoff_bw_days(begin_month_day='04-01', end_month_day='06-30', alpha=.05)
        # print(f'Mann-Kendall Test Results looking for trend in discharge from {s.begin_month_day} to {s.end_month_day}: {s.volume_bw_days_mann_kendall_test}')
    #     runoff_bw_dates_plot2 = streamflow_stat_plots.Plot_Runoff_Volume_Between_2Days(s)
    #     runoff_bw_dates_plot2.plot_runoff_volume_between_2days(site[0])
        mann_kendall_results['Runoff Season Vol'] = s.volume_bw_days_mann_kendall_test
        print('-------------------------------------------------')            
    #     for day in ['01-01', '03-01', '05-01', '08-01']:
        s.calc_runoff_bw_days(begin_month_day='07-01', end_month_day='9-30', alpha=.05)
        # print(f'Mann-Kendall Test Results looking for trend in discharge from {s.begin_month_day} to {s.end_month_day}: {s.volume_bw_days_mann_kendall_test}')
    #     runoff_bw_dates_plot3 = streamflow_stat_plots.Plot_Runoff_Volume_Between_2Days(s)
        mann_kendall_results['Summer Vol'] = s.volume_bw_days_mann_kendall_test

    #     runoff_bw_dates_plot3.plot_runoff_volume_between_2days(site[0])
       
        
        mann_kendall_results['Q on date of peak Q date MK Test'] = s.rolling_yr_Qmax_mk_test
        mann_kendall_results['dayofyear on date of peak Q date MK Test'] = s.rolling_yr_DOYmax_mk_test
        mann_kendall_results['Threshold Vol MK Test'] = s.threshold_vol_mann_kendall_test
        mann_kendall_results['Threshold Vol DOY MK Test'] = s.threshold_vol_dates_mann_kendall_test
        mann_kendall_results['Total Vol MK Test'] = s.total_volume_mann_kendall_test

        volume_stats_for_vip_sites[site] = mann_kendall_results
    except Exception as e:
        raise ValueError('Saving to volume_stats_for_vip_sites prob failed')
        print(e)
    
    sites = []
    site_name = []
    lat = []
    long = []
    # basins = []
    
    PeakRunoffQ_trend = []
    PeakRunoffQ_h = []
    PeakRunoffQ_p = []
    PeakRunoffQ_tau = []
    
    PeakRunoffDOY_trend = []
    PeakRunoffDOY_h = []
    PeakRunoffDOY_p = []
    PeakRunoffDOY_tau = []
    
    ThresholdVol_trend = []
    ThresholdVol_h = []
    ThresholdVol_p = []
    ThresholdVol_tau = []
    
    ThresholdDOY_trend = []
    ThresholdDOY_p = []
    ThresholdDOY_tau = []
    
    ThresholdQ_trend = []
    ThresholdQ_h = []
    ThresholdQ_p = []
    ThresholdQ_tau = []
    
    TotalVol_trend = []
    TotalVol_h = []
    TotalVol_p = []
    TotalVol_tau = []
    
    RunoffVol_trend = []
    RunoffVol_h = []
    RunoffVol_p = []
    RunoffVol_tau = []
    
    SummerVol_trend = []
    SummerVol_h = []
    SummerVol_p = []
    SummerVol_tau = []
    
    FallVol_trend = []
    FallVol_h = []
    FallVol_p = []
    FallVol_tau = []
    
    WinterVol_trend = []
    WinterVol_h = []
    WinterVol_p = []
    WinterVol_tau = []
    
    for site, value in volume_stats_for_vip_sites.items():
        
        sites.append(site)
        site_name.append(value.get('name'))
        lat.append(value.get('lat'))
        long.append(value.get('long'))              
                         
        # basins.append(site[1]['Basin'])
        
        PeakRunoffQ_trend.append(value.get('Q on date of peak Q date MK Test')[0])
        PeakRunoffQ_p.append(value.get('Q on date of peak Q date MK Test')[2])
        PeakRunoffQ_tau.append(value.get('Q on date of peak Q date MK Test')[4])
        
        PeakRunoffDOY_trend.append(value.get('Q on date of peak Q date MK Test')[0])
        PeakRunoffDOY_p.append(value.get('Q on date of peak Q date MK Test')[2])
        PeakRunoffDOY_tau.append(value.get'Q on date of peak Q date MK Test')[4])    
            
        ThresholdVol_trend.append(value.get('Threshold Vol MK Test')[0])
        ThresholdVol_p.append(value.get('Threshold Vol MK Test')[2])
        ThresholdVol_tau.append(value.get('Threshold Vol MK Test')[4])    
            
        ThresholdDOY_trend.append(value.get('Threshold Vol DOY MK Test')[0])
        ThresholdDOY_p.append(value.get('Threshold Vol DOY MK Test')[2])
        ThresholdDOY_tau.append(value.get('Threshold Vol DOY MK Test')[4])            
            
        TotalVol_trend.append(value.get('Total Vol MK Test')[0])
        TotalVol_p.append(value.get('Total Vol MK Test')[2])
        TotalVol_tau.append(value.get('Total Vol MK Test')[4])
        
        SummerVol_trend.append(value.get('Summer Vol')[0])
        SummerVol_p.append(value.get('Summer Vol')[2])
        SummerVol_tau.append(value.get('Summer Vol')[4])
    
        RunoffVol_trend.append(value.get('Runoff Season Vol')[0])
        RunoffVol_p.append(value.get('Runoff Season Vol')[2])
        RunoffVol_tau.append(value.get('Runoff Season Vol')[4])
        
        WinterVol_trend.append(value.get('Winter Vol')[0])
        WinterVol_p.append(value.get('Winter Vol')[2])
        WinterVol_tau.append(value.get('Winter Vol')[4])
    
        FallVol_trend.append(value.get('Fall Vol')[0])
        FallVol_p.append(value.get('Fall Vol')[2])
        FallVol_tau.append(value.get('Fall Vol')[4])
    

    peak_runoffQ = pd.DataFrame({'sites':sites, 'site_name': site_name, 'lat': lat, 'long': long, 'trend': PeakRunoffQ_trend,  'p': PeakRunoffQ_p, 'tau': PeakRunoffQ_tau})
    peak_runoffDOY = pd.DataFrame({'sites':sites, 'site_name': site_name, 'lat': lat, 'long': long, 'trend': PeakRunoffDOY_trend,  'p': PeakRunoffDOY_p, 'tau': PeakRunoffDOY_tau})
    ThresholdVol = pd.DataFrame({'sites':sites, 'site_name': site_name, 'lat': lat, 'long': long, 'trend': ThresholdVol_trend, 'p': ThresholdVol_p, 'tau': ThresholdVol_tau})
    ThresholdDOY = pd.DataFrame({'sites':sites, 'site_name': site_name, 'lat': lat, 'long': long, 'trend': ThresholdDOY_trend, 'p': ThresholdDOY_p, 'tau': ThresholdDOY_tau})
    total_vol = pd.DataFrame({'sites':sites, 'site_name': site_name, 'lat': lat, 'long': long, 'trend': TotalVol_trend, 'p': TotalVol_p, 'tau': TotalVol_tau})
    RunoffVol = pd.DataFrame({'sites':sites, 'site_name': site_name, 'lat': lat, 'long': long, 'trend': RunoffVol_trend, 'p': RunoffVol_p, 'tau': RunoffVol_tau})
    SummerVol = pd.DataFrame({'sites':sites, 'site_name': site_name, 'lat': lat, 'long': long, 'trend': SummerVol_trend, 'p': SummerVol_p, 'tau': SummerVol_tau})
    WinterVol = pd.DataFrame({'sites':sites, 'site_name': site_name, 'lat': lat, 'long': long, 'trend': WinterVol_trend, 'p': WinterVol_p, 'tau': WinterVol_tau})
    FallVol = pd.DataFrame({'sites':sites, 'site_name': site_name, 'lat': lat, 'long': long, 'trend': FallVol_trend, 'p': FallVol_p, 'tau': FallVol_tau})
    
    peak_runoffQ_title = 'Trend in Peak Runoff Volume at Sites with Modified Flows: Mann-Kendall Test Results' 
    peak_runoffDOY_title = 'Trend in Timing During Peak Runoff at Sites with Modified Flows: Mann-Kendall Test Results'
    ThresholdDOY_title = 'Trends in Timing of 50% Yearly Volume at Sites with Modified Flows: Mann-Kendall Test Results'
    ThresholdVol_title = 'Trend in Volume at 50% Yearly Flow at Sites with Modified Flows: Mann-Kendall Test Results'
    total_vol_title = 'Yearly Total Volume Trends at Sites with Modified Flows: Mann-Kendall Test Results'
    SummerVol_title = 'Trend in Late Summer Volumes (08-01 through 09-30) at Sites with Modified Flows: Mann-Kendall Test Results'
    FallVol_title = 'Trend in Fall to early Winter Volumes (10-01 through 12-31) at Sites with Modified Flows: Mann-Kendall Test Results'
    WinterVol_title = 'Trend in Wintertime Volumes (01-01 through 4-01) at Sites with Modified Flows: Mann-Kendall Test Results'
    RunoffVol_title = 'Trend in Runoff Season Volumes (04-01 through 07-31) at Sites with Modified Flows: Mann-Kendall Test Results'
    


    figs_scatter = []
    import plotly.express as px
    fig = px.scatter(peak_runoffQ, 'p', 'tau', color='trend',
                     title=peak_runoffQ_title,
                     hover_data={
                         'Site Name': peak_runoffQ['site_name'], 
                         'Site': peak_runoffQ['sites'], 'Trend': peak_runoffQ['trend'],},
                     symbol = peak_runoffQ['trend'],
                     symbol_map={'no trend':'circle-open', 'decreasing': 'triangle-down', 'increasing':'triangle-up'},
                     color_discrete_map={'decreasing':'red', 'increasing':'green', 'no trend': 'blue'},
                     opacity=0.5

                    )
    figs_scatter.append(fig)
    fig.show()

    fig = px.scatter(peak_runoffDOY, 'p', 'tau', color='trend',
                     title=peak_runoffDOY_title,
                     hover_data={
                         'Site Name': peak_runoffDOY['site_name'], 
                         'Site': peak_runoffDOY['sites'], 'Trend': peak_runoffDOY['trend'],},
                     symbol = peak_runoffDOY['trend'],
                     symbol_map={'no trend':'circle-open', 'decreasing': 'triangle-down', 'increasing':'triangle-up'},
                     color_discrete_map={'decreasing':'red', 'increasing':'green', 'no trend': 'blue'},
                     opacity=0.5

                    )
    figs_scatter.append(fig)
    fig.show()


    fig = px.scatter(ThresholdVol, 'p', 'tau', color='trend',
                     title=ThresholdVol_title,
                     hover_data={
                         'Site Name': ThresholdVol['site_name'], 
                         'Site': ThresholdVol['sites'], 'Trend': ThresholdVol['trend'],},
                     symbol = ThresholdVol['trend'],
                     symbol_map={'no trend':'circle-open', 'decreasing': 'triangle-down', 'increasing':'triangle-up'},
                     color_discrete_map={'decreasing':'red', 'increasing':'green', 'no trend': 'blue'},
                     opacity=0.5
                    )
    figs_scatter.append(fig)
    fig.show()
    fig = px.scatter(ThresholdDOY, 'p', 'tau', color='trend',
                     title=ThresholdDOY_title,
                     hover_data={
                         'Site Name': ThresholdDOY['site_name'], 
                         'Site': ThresholdDOY['sites'], 'Trend': ThresholdDOY['trend'],},
                     symbol = ThresholdDOY['trend'],
                     symbol_map={'no trend':'circle-open', 'decreasing': 'triangle-down', 'increasing':'triangle-up'},
                     color_discrete_map={'decreasing':'red', 'increasing':'green', 'no trend': 'blue'},
                     opacity=0.5
                    )
    figs_scatter.append(fig)
    fig.show()
    # with open('HeadwaterSites_MK_Results2.html', 'a') as f:
    #     f.write(fig.to_html(full_html=False, include_plotlyjs='cdn', default_width=1200, default_height=400))

    fig = px.scatter(total_vol, 'p', 'tau', color='trend',
                     title=total_vol_title,
                     hover_data={
                         'Site Name': total_vol['site_name'], 
                         'Site': total_vol['sites'], 'Trend': total_vol['trend'],},
                     symbol = total_vol['trend'],
                     symbol_map={'no trend':'circle-open', 'decreasing': 'triangle-down', 'increasing':'triangle-up'},
                     color_discrete_map={'decreasing':'red', 'increasing':'green', 'no trend': 'blue'},
                     opacity=0.5
                    )
    figs_scatter.append(fig)
    fig.show()

    fig = px.scatter(SummerVol, 'p', 'tau', color='trend',
                     title=SummerVol_title,
                     hover_data={
                         'Site Name': SummerVol['site_name'], 
                         'Site': SummerVol['sites'], 'Trend': SummerVol['trend'],},
                     symbol = SummerVol['trend'],
                     symbol_map={'no trend':'circle-open', 'decreasing': 'triangle-down', 'increasing':'triangle-up'},
                     color_discrete_map={'decreasing':'red', 'increasing':'green', 'no trend': 'blue'},
                     opacity=0.5
                    )
    figs_scatter.append(fig)
    fig.show()

    fig = px.scatter(FallVol, 'p', 'tau', color='trend',
                     title=FallVol_title,
                     hover_data={
                         'Site Name': FallVol['site_name'], 
                         'Site': FallVol['sites'], 'Trend': FallVol['trend'],},
                     symbol = FallVol['trend'],
                     symbol_map={'no trend':'circle-open', 'decreasing': 'triangle-down', 'increasing':'triangle-up'},
                     color_discrete_map={'decreasing':'red', 'increasing':'green', 'no trend': 'blue'}
                    )
    figs_scatter.append(fig)
    fig.show()

    fig = px.scatter(WinterVol, 'p', 'tau', color='trend',
                     title=WinterVol_title,
                     hover_data={
                         'Site Name': WinterVol['site_name'], 
                         'Site': WinterVol['sites'], 'Trend': WinterVol['trend'],},
                     symbol = WinterVol['trend'],
                     symbol_map={'no trend':'circle-open', 'decreasing': 'triangle-down', 'increasing':'triangle-up'},
                     color_discrete_map={'decreasing':'red', 'increasing':'green', 'no trend': 'blue'},
                     opacity=0.5
                    )
    figs_scatter.append(fig)
    fig.show()

    fig = px.scatter(RunoffVol, 'p', 'tau', color='trend',
                     title=RunoffVol_title,
                     hover_data={
                         'Site Name': RunoffVol['site_name'], 
                         'Site': RunoffVol['sites'], 'Trend': RunoffVol['trend'],},
                     symbol = RunoffVol['trend'],
                     symbol_map={'no trend':'circle-open', 'decreasing': 'triangle-down', 'increasing':'triangle-up'},
                     color_discrete_map={'decreasing':'red', 'increasing':'green', 'no trend': 'blue'},
                     opacity=0.5
                    )
    figs_scatter.append(fig)
    fig.show()

    if os.path.exists(file_name):
        os.remove(file_name)
    for fig in figs_scatter:
        with open(file_name, 'a') as f:
            f.write(fig.to_html(full_html=False, include_plotlyjs='cdn', default_width=1200, default_height=400))
    print(f'Plots have been printed from {start_date} to {end_date}.  Filename: {file_name}.')


    figs = []
    #      dates_ranges = {'01-01':'04-01', '04-01':'07-31', '08-01':'09-30', '10-01':'12-31'} 
    for stat in [peak_runoffQ, peak_runoffDOY, ThresholdVol, ThresholdDOY, total_vol, RunoffVol, SummerVol, FallVol, WinterVol]:
        if stat is peak_runoffQ:
            title=peak_runoffQ_title
        elif stat is peak_runoffDOY:
            title=peak_runoffDOY_title
        elif stat is ThresholdVol:
            title=ThresholdVol_title
        elif stat is ThresholdDOY:
            title=ThresholdDOY_title
        elif stat is total_vol:
            title=total_vol_title
        elif stat is RunoffVol:
            title=RunoffVol_title
        elif stat is SummerVol:
            title=SummerVol_title
        elif stat is WinterVol:
            title=WinterVol_title
        elif stat is FallVol:
            title=FallVol_title


        # options = [['decreasing','circle'], ['no trend', 'circle-open'], ['increasing', 'circle']]
        trends = ['decreasing', 'no trend', 'increasing']
        markers = ['circle', 'circle-stroked', 'information']
        colors = ['red', 'grey', 'blue']


        fig = go.Figure()
        for trend, color in zip(trends, colors):

            fig.add_trace(
               go.Scattermapbox(
                    mode='markers+text',
                    lon = stat[stat["trend"]==trend]['long'],
                    lat = stat[stat["trend"]==trend]['lat'],
                    name = trend,
                    text = stat[stat["trend"]==trend]['site_name'],
                    textposition='top right',
                    textfont=dict(
                        size=20,
                        color='black'
                    ),
                    customdata=stat[stat["trend"]==trend][['sites','p','tau','trend','site_name']],
                    hovertemplate = 
                       '<b>%{text}</b><br>ID : %{customdata[0]}<br>Trend: %{customdata[3]}<br>P-value: %{customdata[1]:.2f}<br>tau: %{customdata[2]:.2f}<br><extra></extra>', # extra removes trace information

                    marker=go.scattermapbox.Marker(
                        size=10,
                        color=color, #stat['p'],
                        colorscale='Viridis',
                        # cmin=1,
                        # cmax=3,
                        showscale=True
                        # symbol=marker, 
            )))

            fig.update_layout(

                mapbox_layers=[
                    {
                        "below": 'traces',
                        "sourcetype": "raster",
                        "sourceattribution": "United States Geological Survey",
                        "source": [
                            "https://basemap.nationalmap.gov/arcgis/rest/services/USGSHydroCached/MapServer/tile/{z}/{y}/{x}"
                            # Options available - replace tag between services/ and /Mapserver with:
                            # USGSHydroCached
                            # USGSImageryOnly
                            # USGSImageryTopo
                            # USGSShadedReliefOnly
                            # USGSTNMBlank
                            # USGSTopo  
                        ],
                        # 'type':'raster',
                        # 'paint':{
                        #     'raster-opacity':0.5
                        #     },
                    }
                    # {
                    #     "below": 'traces',
                    #     "sourcetype": "raster",
                    #     "sourceattribution": "United States Geological Survey",
                    #     "source": [
                    #         "https://basemap.nationalmap.gov/arcgis/rest/services/USGSShadedReliefOnly/MapServer/tile/{z}/{y}/{x}"
                    #         # Options available - replace tag between services/ and /Mapserver with:
                    #         # USGSHydroCached
                    #         # USGSImageryOnly
                    #         # USGSImageryTopo
                    #         # USGSShadedReliefOnly
                    #         # USGSTNMBlank
                    #         # USGSTopo  
                    #     ]       
                    # }
                ],        
                mapbox = {
                    # 'accesstoken': mapbox_token,
                    'style': 'carto-positron', #'stamen-terrain',
                    'zoom':5,
                    'center': go.layout.mapbox.Center(lat=46.1, lon=-117.5)
                },
                title=f'<b>{title}</b>',
                showlegend=False,
                autosize=False,
                width=900,
                height=700
            )
        # file_name = f'{stat=}'.split('=')[0]
        # print(file_name)
        fig.show()
        figs.append(fig)

    file_name = f'headwater_mk_results_maps_{start_date}to{end_date}.html'
    if os.path.exists(file_name):
        os.remove(file_name)
    for figure in figs:
        with open(file_name, 'a') as f:
            f.write(figure.to_html(full_html=False, include_plotlyjs='cdn', default_width=1200, default_height=400))

In [ ]:
headwater[headwater.index==12301250].iat[0,1]

In [ ]:
mann_kendall_results.get('Total Vol MK Test')